# Лабораторна 6. Власна RAG-система: питання українською до документації Sentence Transformers

Система відповідає на питання про бібліотеку **Sentence Transformers** (версія 6.3.0) з посиланням на джерело: файл документації, розділ і фрагмент тексту. Питання ставляться **українською**, документація **англійською**, тож система має справлятися з міжмовним пошуком.

Ноутбук проходить усі кроки завдання 1:

1. **1.1 Корпус** — що за документи, звідки, яка ліцензія, чому потрібен саме RAG.
2. **1.2 Бенчмарк** — 30 власних питань (25 з відповіддю, 5 без) з еталонною відповіддю та еталонним фрагментом.
3. **1.3 Індексація** — модель ембедингів, розбиття на фрагменти, векторне сховище.
4. **1.4 Ретрив** — hit@k і MRR для кількох конфігурацій.
5. **1.5 Генерація** — відповідь із цитуванням і відмовою; правильність відповідей і цитувань.
6. **1.6 Аналіз відмов** — розбір помилок: тип, причина, виправлення.
7. **1.7 Інтерфейс** — запит питання й показ джерел.
8. **1.8 Звіт** — підсумок.

**Як запускати.** Colab з GPU T4 (Runtime → Change runtime type → T4 GPU), далі Runtime → Run all. Повний прогін займає орієнтовно 30–40 хвилин (завантаження моделей, індексація, генерація). Ані вагів моделей, ані векторного індексу в репозиторії немає: усе будується тут. Корпус завантажується з GitHub за фіксованим тегом і комітом (розділ 1.1).

**Генерація.** Відповіді генерує локальна модель Qwen3-4B-Instruct-2507 через `transformers` (так само, як у демо 2 і 3). Лабораторну 4 (OpenAI-сумісний сервер) не виконували, тому окремого сервера немає: інтерфейс генерації (`generate(messages)`) такий самий, як у запиті до чат-моделі, тож його можна замінити на виклик OpenAI-сумісного endpoint без змін решти коду.

In [1]:
# faiss-cpu — векторне сховище, rank_bm25 — лексичний пошук; sentence-transformers дає модель ембедингів і cross-encoder
!pip install -q "sentence-transformers>=5" faiss-cpu rank_bm25

In [2]:
import math
import random
import re
import subprocess
import time
from collections import Counter
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

# Моделі (як у демо 2 і 3)
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Корпус: документація Sentence Transformers за фіксованим тегом
CORPUS_REPO = "https://github.com/huggingface/sentence-transformers"
CORPUS_TAG = "v6.3.0"
SRC_DIR = Path("st_src")

Пристрій: cuda


## 1.1. Корпус

**Що це.** Офіційна документація бібліотеки [Sentence Transformers](https://github.com/huggingface/sentence-transformers) (каталог `docs/` репозиторію, тег `v6.3.0`). Бібліотеку використовують у демо-ноутбуках цієї лабораторної (ембединги, cross-encoder для переранжування), а її документація велика, англомовна й швидко змінюється між версіями.

**Хто й навіщо питатиме.** Україномовний розробник чи студент, який налаштовує ембединг-пошук, переранжування чи донавчання моделей. Типові питання: «як тепер називається параметр…», «які залежності потрібні…», «яке значення за замовчуванням…», «чим відрізняється X від Y». Питання українською, документація англійською, тож це ще й перевірка міжмовного пошуку.

**Ліцензія.** Apache License 2.0 (файл `LICENSE` у репозиторії; перевіряємо його в коді нижче). Ліцензія дозволяє копіювати й поширювати документацію, але за вимогами лабораторної в репозиторій потрапляють лише ноутбуки, тож корпус **не додається**: ноутбук завантажує його з GitHub за тегом і коміту, які виводяться в клітинці нижче.

**Що входить у корпус.** Усі файли `.md` і `.rst` із `docs/`, крім:
- каталогу `package_reference/`: у ньому лише директиви `autodoc` (`.. autoclass::`), а справжній текст лежить у docstring-ах коду, якого в корпусі немає;
- службових файлів менше 1000 байтів (підписи бокових панелей тощо).

Із тексту прибираємо розмітку без змісту: HTML-картки навігації, зображення, змісти (`toctree`). Код і таблиці залишаються.

Функції нижче завантажують корпус і розбивають його на фрагменти двома способами:

- **`fixed_chunks`** — вікно фіксованої довжини 1000 символів з перекриттям 200 (база з демо 2): не зважає на структуру, межа може пройти посередині коду чи речення;
- **`structured_chunks`** — розбиття **за структурою документа**: спершу на розділи за заголовками (Markdown `#` і reStructuredText з підкресленням), у розділі — на цілі блоки (абзаци, блоки коду, елементи списків), які складаються у фрагмент до заданого розміру. Блок коду не розривається, якщо вміщується у фрагмент. До кожного фрагмента при індексації додається **шлях розділів** (назва файлу > заголовок > підзаголовок), так само, як у демо назва документа дописувалась на початок фрагмента.

In [3]:
CORPUS_EXTENSIONS = {".md", ".rst"}
MIN_FILE_BYTES = 1000                 # відкидаємо службові файли на кілька рядків
EXCLUDED_DIRS = {"package_reference"}  # лише директиви autodoc: тексту немає, він у docstring-ах коду

HTML_TAG = re.compile(r"</?(?:div|a|br|p|span|b|i|iframe|details|summary|img|code|center)\b[^>]*>", re.IGNORECASE)
RST_UNDERLINE = re.compile(r"^([=\-~^\"*+#`'.:_])\1{2,}\s*$")
MD_HEADING = re.compile(r"^(#{1,6})\s+(.*\S)\s*$")


def list_corpus_files(docs_root):
    files = []
    for path in sorted(Path(docs_root).rglob("*")):
        relative = path.relative_to(docs_root)
        if path.suffix not in CORPUS_EXTENSIONS or EXCLUDED_DIRS & set(relative.parts):
            continue
        if path.stat().st_size >= MIN_FILE_BYTES:
            files.append(path)
    return files


def clean_text(text):
    """Прибираємо розмітку, яка не несе змісту: HTML-картки, зображення, зміст (toctree)."""
    text = text.replace("\r\n", "\n")
    lines, skip_indent = [], None
    for line in text.split("\n"):
        stripped = line.strip()
        indent = len(line) - len(line.lstrip())
        if re.match(r"\.\. (raw:: html|toctree::|image::|figure::)", stripped):
            skip_indent = indent          # пропускаємо директиву і весь її вкладений вміст
            continue
        if skip_indent is not None:
            if stripped == "" or indent > skip_indent:
                continue
            skip_indent = None
        lines.append(HTML_TAG.sub("", line))
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def split_sections(text):
    """Ділить документ на розділи за заголовками (Markdown # і reStructuredText з підкресленням).
    Повертає список (шлях_заголовків, текст_розділу)."""
    lines = text.split("\n")
    sections, stack, body = [], [], []
    rst_levels, in_fence = {}, False

    def flush():
        content = "\n".join(body).strip()
        if content:
            sections.append((" > ".join(title for _, title in stack), content))
        body.clear()

    def open_heading(level, title):
        flush()
        while stack and stack[-1][0] >= level:
            stack.pop()
        stack.append((level, title))

    i = 0
    while i < len(lines):
        line = lines[i]
        if line.lstrip().startswith(("```", "~~~")):
            in_fence = not in_fence
        heading = MD_HEADING.match(line) if not in_fence else None
        next_line = lines[i + 1] if i + 1 < len(lines) else ""
        if heading:
            open_heading(len(heading.group(1)), heading.group(2))
        elif (not in_fence and line.strip() and not line.startswith((" ", "\t", ".."))
              and RST_UNDERLINE.match(next_line) and len(next_line.strip()) >= len(line.strip())):
            char = next_line.strip()[0]
            level = rst_levels.setdefault(char, len(rst_levels) + 1)
            open_heading(level, line.strip())
            i += 1
        else:
            body.append(line)
        i += 1
    flush()
    return sections


def split_blocks(text):
    """Абзаци й блоки коду. Порожній рядок не завершує блок, якщо далі йде відступ (код, продовження списку) або ми всередині ```."""
    blocks, current, in_fence = [], [], False
    lines = text.split("\n")
    for index, line in enumerate(lines):
        if line.lstrip().startswith(("```", "~~~")):
            in_fence = not in_fence
        if line.strip() == "" and not in_fence:
            following = next((candidate for candidate in lines[index + 1:] if candidate.strip()), "")
            if current and not following.startswith(("   ", "\t")):
                blocks.append("\n".join(current))
                current = []
                continue
        current.append(line)
    if current:
        blocks.append("\n".join(current))
    return [block.strip("\n") for block in blocks if block.strip()]


def split_long_block(block, size):
    """Блок довший за size ріжемо за рядками, а надто довгий рядок — за довжиною."""
    parts, current = [], ""
    for line in block.split("\n"):
        pieces = [line[i:i + size] for i in range(0, len(line), size)] or [""]
        for piece in pieces:
            if current and len(current) + 1 + len(piece) > size:
                parts.append(current)
                current = piece
            else:
                current = f"{current}\n{piece}" if current else piece
    if current:
        parts.append(current)
    return parts


def fixed_chunks(doc_id, text, size=1000, overlap=200):
    """База: вікно фіксованої довжини з перекриттям (як у демо 2), без урахування структури."""
    return [{"doc_id": doc_id, "section": "", "text": text[start:start + size]}
            for start in range(0, max(len(text) - overlap, 1), size - overlap)]


def structured_chunks(doc_id, text, size=1000):
    """Фрагменти за структурою: у межах розділу складаємо цілі блоки (абзаци, код) до size символів."""
    chunks = []
    for section, content in split_sections(text):
        pieces = []
        for block in split_blocks(content):
            pieces += split_long_block(block, size) if len(block) > size else [block]
        current = ""
        for piece in pieces:
            if current and len(current) + 2 + len(piece) > size:
                chunks.append({"doc_id": doc_id, "section": section, "text": current})
                current = piece
            else:
                current = f"{current}\n\n{piece}" if current else piece
        if current:
            chunks.append({"doc_id": doc_id, "section": section, "text": current})
    return chunks


def chunk_text_for_index(chunk):
    """Текст, який кодується в індексі: назва документа й шлях розділів + сам фрагмент."""
    header = chunk["doc_id"] + (" > " + chunk["section"] if chunk["section"] else "")
    return f"{header}\n{chunk['text']}"

In [4]:
if not SRC_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", CORPUS_TAG, CORPUS_REPO + ".git", str(SRC_DIR)], check=True)
COMMIT = subprocess.run(["git", "-C", str(SRC_DIR), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
DOCS_ROOT = SRC_DIR / "docs"
licence_text = (SRC_DIR / "LICENSE").read_text(encoding="utf-8")
is_apache = "Apache License" in licence_text and "Version 2.0" in licence_text
print(f"Джерело: {CORPUS_REPO}, тег {CORPUS_TAG}, коміт {COMMIT}")
print(f"Файл LICENSE містить Apache License, Version 2.0: {is_apache}")

corpus_files = list_corpus_files(DOCS_ROOT)
raw_texts = {str(path.relative_to(DOCS_ROOT)): path.read_text(encoding="utf-8") for path in corpus_files}
documents = {doc_id: clean_text(text) for doc_id, text in raw_texts.items()}

total_bytes = sum(path.stat().st_size for path in corpus_files)
total_chars = sum(len(text) for text in documents.values())
formats = Counter(path.suffix for path in corpus_files)
print(f"\nДокументів: {len(documents)} ({', '.join(f'{ext}: {n}' for ext, n in sorted(formats.items()))})")
print(f"Обсяг: {total_bytes / 1024:.0f} КБ вихідного тексту, {total_chars / 1024:.0f} тис. символів після очищення")

rows = []
for doc_id, text in documents.items():
    top = doc_id.split("/")[0] if "/" in doc_id else "(корінь docs/)"
    rows.append({"каталог": top, "файлів": 1, "КБ (вихідний)": len(raw_texts[doc_id]) / 1024, "тис. симв. (очищений)": len(text) / 1000})
display(pd.DataFrame(rows).groupby("каталог").sum().round(1))

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
corpus_tokens = sum(len(tokenizer(text)["input_ids"]) for text in documents.values())
corpus_tokens_text = f"{corpus_tokens:,}".replace(",", " ")
print(f"\nВесь корпус у токенах генеративної моделі: {corpus_tokens_text}")

# Скільки памʼяті зайняв би KV-кеш, якби весь корпус вкласти в один промпт (fp16: 2 байти, ключі й значення для кожного шару)
config = AutoConfig.from_pretrained(GENERATOR_MODEL_NAME)
head_dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads
kv_bytes_per_token = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2
kv_cache_gb = corpus_tokens * kv_bytes_per_token / 2 ** 30
print(f"KV-кеш на один токен: {kv_bytes_per_token / 1024:.0f} КБ; для всього корпусу: {kv_cache_gb:.0f} ГБ (на T4 доступно ~15 ГБ, з яких ~8 ГБ займають ваги моделі)")

Джерело: https://github.com/huggingface/sentence-transformers, тег v6.3.0, коміт cddcaee3405a5efde83c1d98f89c6ebd6733dab6
Файл LICENSE містить Apache License, Version 2.0: True

Документів: 44 (.md: 27, .rst: 17)
Обсяг: 842 КБ вихідного тексту, 722 тис. символів після очищення


,файлів,КБ (вихідний),тис. симв. (очищений)
каталог,,,
(корінь docs/),5,148.4,151.9
cross_encoder,6,171.6,141.5
multi_vector_encoder,7,150.6,126.7
pretrained-models,10,25.9,26.5
sentence_transformer,11,219.9,200.3
sparse_encoder,5,124.8,92.6


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            


Весь корпус у токенах генеративної моделі: 169 234
KV-кеш на один токен: 144 КБ; для всього корпусу: 23 ГБ (на T4 доступно ~15 ГБ, з яких ~8 ГБ займають ваги моделі)


**Чому для цієї задачі потрібен саме RAG, а не промпт чи донавчання.**

- **Не вміщується в промпт.** Весь корпус — це понад сотня тисяч токенів (точне число виведено вище). Модель формально має довгий контекст, але вкласти корпус у кожен запит не вдасться на GPU з 15 ГБ: лише KV-кеш такого контексту (його розмір пораховано вище за конфігурацією моделі) перевищує всю відеопамʼять, а час відповіді зростав би на порядки. RAG кладе в промпт лише 5 найрелевантніших фрагментів (у прогоні в середньому близько 1,5 тис. токенів разом із системним промптом і питанням).
- **Документація змінюється.** У версіях 5.4 і 6.0 перейменовано методи й параметри та піднято мінімальні версії `transformers` і `torch` (розділ «Migration Guide» самого корпусу). Донавчена модель застаріє після наступного релізу й не скаже, звідки взяла відповідь; індекс RAG перебудовується за хвилини.
- **Потрібне джерело.** Відповідь про назву параметра чи версію треба вміти перевірити, а RAG повертає посилання на файл, розділ і фрагмент.
- **Модель без документів може не знати відповіді.** Версія 6.3.0 новіша за навчальні дані генеративної моделі, тож без контексту вона може відповідати за старою версією бібліотеки або вигадувати. Це перевіряємо експериментально: у розділі 1.5 є прогін «без RAG» на тих самих питаннях.

## 1.2. Бенчмарк

Набір із **30 питань**, складених за текстом документації **до** налаштування системи: 25 питань мають відповідь у документах, 5 — ні. Кожне питання має:

- **`kind`** — тип: `число`, `ідентифікатор` (назва параметра, методу, класу), `концепція`, `без відповіді`;
- **`gold`** — еталонні документи (де саме є відповідь);
- **`evidence`** — вирази-докази (регулярні вирази, за якими впізнається **еталонний фрагмент**: фрагмент з еталонного документа, що містить такий вираз). Перевіряти лише документ замало: у довгих файлах десятки фрагментів, а відповідь є лише в одному;
- **`facts`** — ключові факти, які мусить містити правильна відповідь (регулярні вирази; відповідь українською, але імена параметрів і числа збігаються з документацією);
- **`reference`** — еталонна відповідь.

Якщо один і той самий абзац є в документації кількох бібліотек (наприклад, однаково написаний абзац у `training_overview.md` усіх чотирьох типів моделей), еталонними вважаються **всі** файли, де він є; список `gold` перевіряється кодом автоматично.

Питання без відповіді поділяються на «далекі від корпусу» (TPU, Rust) і «близькі» (у документах є схожа, але інша інформація: інша модель, інша кількість GPU, умовна ліцензія з прикладу): саме на таких системі найлегше вигадати відповідь. Для кожного з них у коді задано **пробу відсутності**: регулярний вираз, який не повинен знаходитись у жодному документі корпусу.

In [5]:
BENCHMARK = [
    dict(id="q01", kind="число",
         question="Яку мінімальну версію Python рекомендують для Sentence Transformers?",
         gold=["installation.md", "migration_guide.md"],
         evidence=[r"Python 3\.10\+", r"minimum Python version stays 3\.10"],
         facts=[r"3\.10"],
         reference="Python 3.10 або новіша (рекомендовано Python 3.10+)."),
    dict(id="q02", kind="число",
         question="Яку версію бібліотеки transformers вимагає Sentence Transformers v6.0?",
         gold=["installation.md", "migration_guide.md"],
         evidence=[r"requires `transformers` v5\.x", r"transformers v5\.0\+"],
         facts=[r"5\.x|5\.0|\bv?5\b"],
         reference="transformers v5.x (раніше потрібна була 4.41+)."),
    dict(id="q03", kind="ідентифікатор",
         question="Як після оновлення до v5.4+ називається метод get_sentence_embedding_dimension() у SentenceTransformer?",
         gold=["migration_guide.md"],
         evidence=[r"SentenceTransformer\.get_embedding_dimension\(\)"],
         facts=[r"get_embedding_dimension"],
         reference="get_embedding_dimension()."),
    dict(id="q04", kind="ідентифікатор",
         question="Який параметр замінив tokenizer_kwargs при створенні моделей Sentence Transformers?",
         gold=["migration_guide.md"],
         evidence=[r"has been renamed to ``processor_kwargs``"],
         facts=[r"processor_kwargs"],
         reference="processor_kwargs (старе ім'я ще працює, але видає попередження)."),
    dict(id="q05", kind="ідентифікатор",
         question="Як перейменовано параметр tokenizer у SentenceTransformerTrainer, починаючи з версії 5.4?",
         gold=["migration_guide.md"],
         evidence=[r"has been renamed to ``processing_class``"],
         facts=[r"processing_class"],
         reference="Параметр tokenizer став processing_class."),
    dict(id="q06", kind="концепція",
         question="Який режим пулінгу за замовчуванням використовують CausalLM-моделі, такі як Llama чи Qwen, починаючи з версії 5.4?",
         gold=["migration_guide.md"],
         evidence=[r"now default to last token pooling"],
         facts=[r"(?i)останн\w*\s+токен|last[\s-]*token"],
         reference="Пулінг за останнім токеном (раніше завжди застосовувався mean pooling)."),
    dict(id="q07", kind="число",
         question="У скільки разів DDP прискорює навчання порівняно з відсутністю паралелізму за таблицею порівняння?",
         gold=["sentence_transformer/training/distributed.rst"],
         evidence=[r"2\.562x speedup"],
         facts=[r"2[.,]56"],
         reference="Приблизно у 2,56 раза: 6980 проти 2724 зразків за секунду."),
    dict(id="q08", kind="число",
         question="Яке медіанне прискорення порівняно з FP32 дає FP16 разом із Flash Attention та unpadding?",
         gold=["sentence_transformer/usage/efficiency.rst"],
         evidence=[r"3\.87x\s+over FP32"],
         facts=[r"3[.,]87"],
         reference="Приблизно у 3,87 раза (медіана за протестованими моделями)."),
    dict(id="q09", kind="концепція",
         question="Які бекенди для обчислення ембедингів підтримує Sentence Transformers?",
         gold=["cross_encoder/usage/efficiency.rst", "multi_vector_encoder/usage/efficiency.rst", "sentence_transformer/usage/efficiency.rst", "sparse_encoder/usage/efficiency.rst"],
         evidence=[r"supports 3 backends"],
         facts=[r"(?i)pytorch|torch", r"(?i)onnx", r"(?i)openvino"],
         reference="Три бекенди: PyTorch (типовий), ONNX і OpenVINO."),
    dict(id="q10", kind="концепція",
         question="Який бекенд радять для CPU, якщо невелика втрата точності прийнятна?",
         gold=["cross_encoder/usage/efficiency.rst", "multi_vector_encoder/usage/efficiency.rst", "sentence_transformer/usage/efficiency.rst", "sparse_encoder/usage/efficiency.rst"],
         evidence=[r"C -->\|yes\| G\[openvino-qint8\]"],
         facts=[r"(?i)openvino"],
         reference="OpenVINO з int8-квантизацією (openvino-qint8)."),
    dict(id="q11", kind="ідентифікатор",
         question="Який оператор використовують multi-vector моделі для підрахунку подібності між запитом і документом?",
         gold=["multi_vector_encoder/usage/kernels.rst", "multi_vector_encoder/usage/usage.rst"],
         evidence=[r"scored with the \*\*MaxSim operator\*\*", r"MaxSim\s+scores a query against a document"],
         facts=[r"(?i)maxsim"],
         reference="MaxSim: для кожного токена запиту береться найбільша подібність до токенів документа, потім значення додаються."),
    dict(id="q12", kind="концепція",
         question="Чому cross-encoder повільніший за bi-encoder і як його зазвичай застосовують?",
         gold=["cross_encoder/usage/usage.rst", "quickstart.rst"],
         evidence=[r"Often \*\*slower\*\* than a Sentence Transformer"],
         facts=[r"(?i)пар|pair", r"(?i)переранж|ре-?ранж|повторн\w*\s+ранж|rerank|re-rank"],
         reference="Він рахується для кожної пари запит–текст, тому повільніший; його використовують, щоб переранжувати top-k результатів bi-encoder."),
    dict(id="q13", kind="ідентифікатор",
         question="Як змусити MS Marco cross-encoder повертати оцінки від 0 до 1?",
         gold=["cross_encoder/usage/usage.rst"],
         evidence=[r"MS Marco models return logits rather than scores between 0 and 1"],
         facts=[r"(?i)sigmoid|сигмоїд"],
         reference="Завантажити CrossEncoder з activation_fn=torch.nn.Sigmoid(); на ранжування це не впливає."),
    dict(id="q14", kind="ідентифікатор",
         question="Які методи рекомендовано використовувати для кодування запитів і документів у задачах пошуку?",
         gold=["sentence_transformer/usage/usage.rst"],
         evidence=[r"are the recommended methods"],
         facts=[r"encode_query", r"encode_document"],
         reference="encode_query і encode_document: вони автоматично застосовують промпти для запитів і документів."),
    dict(id="q15", kind="концепція",
         question="Який пристрій обирає PyTorch-бекенд, якщо device не вказано?",
         gold=["cross_encoder/usage/efficiency.rst", "multi_vector_encoder/usage/efficiency.rst", "sentence_transformer/usage/efficiency.rst"],
         evidence=[r"first available option among"],
         facts=[r"(?i)cuda", r"(?i)cpu"],
         reference="Перший доступний із cuda, mps, npu, xpu, hpu, neuron, cpu."),
    dict(id="q16", kind="ідентифікатор",
         question="Яку додаткову залежність (extra) треба встановити, щоб використовувати use_kernels=True?",
         gold=["multi_vector_encoder/usage/kernels.rst"],
         evidence=[r"sentence-transformers\[kernels\]"],
         facts=[r"kernels"],
         reference="pip install \"sentence-transformers[kernels]\"."),
    dict(id="q17", kind="ідентифікатор",
         question="Яку функцію втрат зазвичай використовують для навчання топових ембединг-моделей на парах (anchor, positive) без міток?",
         gold=["sentence_transformer/loss_overview.md"],
         evidence=[r"(?<![A-Za-z])MultipleNegativesRankingLoss \(a\.k\.a\. InfoNCE or in-batch negatives loss\) is commonly used to train the top performing embedding models"],
         facts=[r"MultipleNegativesRankingLoss|MNRL"],
         reference="MultipleNegativesRankingLoss (InfoNCE, in-batch negatives)."),
    dict(id="q18", kind="ідентифікатор",
         question="Який batch sampler рекомендовано для функцій втрат, що використовують in-batch negatives?",
         gold=["cross_encoder/training_overview.md", "multi_vector_encoder/training_overview.md", "sentence_transformer/training_overview.md", "sparse_encoder/training_overview.md"],
         evidence=[r'losses that use \"in-batch negatives\" benefit from no duplicates'],
         facts=[r"NO_DUPLICATES"],
         reference="BatchSamplers.NO_DUPLICATES."),
    dict(id="q19", kind="число",
         question="Із скількох компонентів складається навчання моделі Sentence Transformer?",
         gold=["cross_encoder/training_overview.md", "multi_vector_encoder/training_overview.md", "sentence_transformer/training_overview.md", "sparse_encoder/training_overview.md"],
         evidence=[r"involves between 4 to 6 components"],
         facts=[r"4\D{1,15}6"],
         reference="Від 4 до 6 компонентів (модель, датасет, функція втрат, аргументи, евалуатор, тренер)."),
    dict(id="q20", kind="ідентифікатор",
         question="Який клас Sentence Transformers замінює pylate.models.ColBERT при міграції з PyLate?",
         gold=["migration_guide.md"],
         evidence=[r"pylate\.models\.ColBERT\(model_name_or_path=\.\.\.\)` \| `MultiVectorEncoder"],
         facts=[r"MultiVectorEncoder"],
         reference="MultiVectorEncoder."),
    dict(id="q21", kind="концепція",
         question="Який тип значень score повертає CrossEncoder.rank у версії 6?",
         gold=["migration_guide.md"],
         evidence=[r"``rank`` now returns Python floats"],
         facts=[r"(?i)float|плаваюч|дійсн"],
         reference="Звичайні числа Python типу float (раніше numpy.float32)."),
    dict(id="q22", kind="концепція",
         question="Чи потрібен trust_remote_code=True для завантаження локальної моделі з власним класом модуля у версії 6?",
         gold=["migration_guide.md"],
         evidence=[r"now always requires `trust_remote_code=True`"],
         facts=[r"(?i)^\W*так\b|(?<!не )завжди"],  # «Так…» або «завжди…»; «не завжди» і «ні» не зараховуються
         reference="Так: тепер завжди потрібен trust_remote_code=True, навіть для локальних каталогів."),
    dict(id="q23", kind="число",
         question="У скільки разів, за документацією, all-MiniLM-L6-v2 швидша за all-mpnet-base-v2?",
         gold=["sentence_transformer/pretrained_models.md"],
         evidence=[r"is 5 times faster"],
         facts=[r"\b5\b|п[’'ʼ]ят"],
         reference="Приблизно у 5 разів швидша, зберігаючи добру якість."),
    dict(id="q24", kind="концепція",
         question="Які обмеження має FSDP під час навчання в Sentence Transformers?",
         gold=["sentence_transformer/training/distributed.rst"],
         evidence=[r"You can't use the ``evaluator`` functionality with FSDP"],
         facts=[r"(?i)evaluator|евалуатор|оцінюван"],
         reference="FSDP підтримується не повністю: не можна використовувати evaluator, а збереження моделі потребує FULL_STATE_DICT."),
    dict(id="q25", kind="концепція",
         question="Чому DDP рекомендують замість DP для розподіленого навчання?",
         gold=["sentence_transformer/training/distributed.rst"],
         evidence=[r"DDP is generally faster than DP"],
         facts=[r"(?i)швидш|faster", r"(?i)менш\w*\s+даних|less data|рівномірн"],
         reference="DDP швидший, бо передає менше даних, рівномірніше розподіляє роботу між GPU і дозволяє навчання на кількох машинах."),
    dict(id="q26", kind="без відповіді",
         question="Як налаштувати Sentence Transformers для роботи на TPU?",
         gold=[],
         evidence=[],
         facts=[],
         reference=None),
    dict(id="q27", kind="без відповіді",
         question="Скільки параметрів має модель all-mpnet-base-v2?",
         gold=[],
         evidence=[],
         facts=[],
         reference=None),
    dict(id="q28", kind="без відповіді",
         question="Скільки зразків за секунду дає DDP-навчання на 8 відеокартах?",
         gold=[],
         evidence=[],
         facts=[],
         reference=None),
    dict(id="q29", kind="без відповіді",
         question="Яка ліцензія у моделі all-MiniLM-L6-v2?",
         gold=[],
         evidence=[],
         facts=[],
         reference=None),
    dict(id="q30", kind="без відповіді",
         question="Як використовувати Sentence Transformers із мовою програмування Rust?",
         gold=[],
         evidence=[],
         facts=[],
         reference=None),
]

In [6]:
ABSENCE_PROBES = {
    "q26": r"\bTPU",                                                           # TPU в документації немає
    "q27": r"all-mpnet-base-v2.{0,150}\bparam",                                 # параметри all-mpnet-base-v2 не названо (є лише «mpnet-base (133M parameters)» — інша модель)
    "q28": r"\b(?:8|eight)\s*(?:x\s*)?(?:GPUs?|V100|A100)",                    # заміри DDP лише на 4 GPU
    "q29": r"all-MiniLM-L6-v2.{0,200}licen[sc]e|licen[sc]e.{0,200}all-MiniLM-L6-v2",  # ліцензія цієї моделі не названа (є лише license=\"apache-2.0\" у прикладах моделей, що навчаються)
    "q30": r"\bRust\b",
}

In [7]:
answerable_items = [item for item in BENCHMARK if item["gold"]]
unanswerable_items = [item for item in BENCHMARK if not item["gold"]]
ITEM_BY_ID = {item["id"]: item for item in BENCHMARK}
ITEM_BY_QUESTION = {item["question"]: item for item in BENCHMARK}

problems = []
for item in answerable_items:
    found = sorted(doc_id for doc_id, text in documents.items() if any(re.search(e, text, re.S) for e in item["evidence"]))
    if found != sorted(item["gold"]):
        problems.append((item["id"], "gold не збігається з документами, де знайдено доказ", found, item["gold"]))
for item in unanswerable_items:
    hits = [doc_id for doc_id, text in documents.items() if re.search(ABSENCE_PROBES[item["id"]], text, re.I | re.S)]
    if hits:
        problems.append((item["id"], "проба відсутності знайшла збіг", hits))
print("Перевірка бенчмарку:", "помилок немає" if not problems else problems)

summary = pd.DataFrame(BENCHMARK).groupby("kind").size().rename("питань").to_frame()
display(summary.T)
display(pd.DataFrame([{"id": b["id"], "тип": b["kind"], "питання": b["question"], "еталонних файлів": len(b["gold"]), "еталонна відповідь": b["reference"] or "— (відмова)"} for b in BENCHMARK]))

Перевірка бенчмарку: помилок немає


kind,без відповіді,концепція,число,ідентифікатор
питань,5,9,6,10


,id,тип,питання,еталонних файлів,еталонна відповідь
0,q01,число,Яку мінімальну версію Python рекомендують для ...,2,Python 3.10 або новіша (рекомендовано Python 3...
1,q02,число,Яку версію бібліотеки transformers вимагає Sen...,2,transformers v5.x (раніше потрібна була 4.41+).
2,q03,ідентифікатор,Як після оновлення до v5.4+ називається метод ...,1,get_embedding_dimension().
3,q04,ідентифікатор,Який параметр замінив tokenizer_kwargs при ств...,1,"processor_kwargs (старе ім'я ще працює, але ви..."
4,q05,ідентифікатор,Як перейменовано параметр tokenizer у Sentence...,1,Параметр tokenizer став processing_class.
5,q06,концепція,Який режим пулінгу за замовчуванням використов...,1,Пулінг за останнім токеном (раніше завжди заст...
6,q07,число,У скільки разів DDP прискорює навчання порівня...,1,"Приблизно у 2,56 раза: 6980 проти 2724 зразків..."
7,q08,число,Яке медіанне прискорення порівняно з FP32 дає ...,1,"Приблизно у 3,87 раза (медіана за протестовани..."
8,q09,концепція,Які бекенди для обчислення ембедингів підтриму...,4,"Три бекенди: PyTorch (типовий), ONNX і OpenVINO."
9,q10,концепція,"Який бекенд радять для CPU, якщо невелика втра...",4,OpenVINO з int8-квантизацією (openvino-qint8).


## 1.3. Індексація

**Модель ембедингів** — [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B): багатомовна (потрібно для міжмовного пошуку «питання українською → документація англійською»), 0,6 млрд параметрів, 1024-вимірні ембединги, запит кодується з інструкцією (`prompt_name="query"`), документ — без неї (як у демо 1–3). Ембединги нормалізуємо, тож скалярний добуток дорівнює косинусній подібності.

**Векторне сховище** — [FAISS](https://github.com/facebookresearch/faiss), індекс `IndexFlatIP` (точний пошук за скалярним добутком). Для 1–2 тис. фрагментів (935–1936 залежно від розбиття) повний перебір працює за мілісекунди, тож наближений індекс не потрібен. Поряд з ним для кожного набору фрагментів будується BM25-індекс (`rank_bm25`) для лексичного пошуку.

**Розбиття** порівнюємо за трьома варіантами: `fixed-1000` (база з демо: вікно 1000 символів, перекриття 200), `struct-1000` і `struct-500` (розбиття за структурою, цільовий розмір 1000 і 500 символів, перекриття немає: межі збігаються з межами блоків). У кожному фрагменті, що кодується, спереду стоїть шлях «файл > розділ > підрозділ».

**Переранжування** — cross-encoder [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3) (багатомовний) застосовується до 20 кандидатів гібридного пошуку.

**Генерація** — [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) (4 млрд параметрів, у половинній точності ~8 ГБ), жадібне декодування (`do_sample=False`), локально. Та сама модель перекладає питання англійською (розділ 1.4) і виступає суддею (розділ 1.5; обмеження такого вибору розглянуто там же).

In [8]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device, model_kwargs={"dtype": dtype})
embedder.max_seq_length = 1024
reranker = CrossEncoder(RERANKER_MODEL_NAME, device=device, model_kwargs={"dtype": dtype})
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)
print(f"Розмірність ембедингів: {embedder.get_sentence_embedding_dimension()}, максимум токенів для ембедингу: {embedder.max_seq_length}")


def generate(messages, max_new_tokens=350):
    # Жадібне декодування, як у демо 2: відповіді відтворювані
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def sync():
    if device == "cuda":
        torch.cuda.synchronize()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (20 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Розмірність ембедингів: 1024, максимум токенів для ембедингу: 1024


/tmp/ipykernel_487/1334081437.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Розмірність ембедингів: {embedder.get_sentence_embedding_dimension()}, максимум токенів для ембедингу: {embedder.max_seq_length}")


In [9]:
CHUNKINGS = {
    "fixed-1000": lambda doc_id, text: fixed_chunks(doc_id, text, size=1000, overlap=200),
    "struct-1000": lambda doc_id, text: structured_chunks(doc_id, text, size=1000),
    "struct-500": lambda doc_id, text: structured_chunks(doc_id, text, size=500),
}
chunk_sets = {name: [chunk for doc_id, text in documents.items() for chunk in split(doc_id, text)] for name, split in CHUNKINGS.items()}


def matches_evidence(chunk, item):
    # Еталонний фрагмент: з еталонного документа і з виразом-доказом у тексті (без шляху розділів)
    return chunk["doc_id"] in item["gold"] and any(re.search(e, chunk["text"], re.S) for e in item["evidence"])


rows = []
for name, chunk_list in chunk_sets.items():
    lengths = [len(c["text"]) for c in chunk_list]
    covered = sum(any(matches_evidence(c, item) for c in chunk_list) for item in answerable_items)
    rows.append({"розбиття": name, "фрагментів": len(chunk_list), "середня довжина": np.mean(lengths), "медіана": np.median(lengths),
                 "макс.": max(lengths), "перекриття, символів": 200 if name.startswith("fixed") else 0,
                 "питань з еталонним фрагментом": f"{covered} з {len(answerable_items)}"})
display(pd.DataFrame(rows).round(0))

,розбиття,фрагментів,середня довжина,медіана,макс.,"перекриття, символів",питань з еталонним фрагментом
0,fixed-1000,935,981.0,1000.0,1000,200,25 з 25
1,struct-1000,994,733.0,834.0,1000,0,25 з 25
2,struct-500,1936,376.0,421.0,500,0,25 з 25


In [10]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())


class Index:
    # Один набір фрагментів: ембединги у FAISS і BM25 над тими самими текстами
    def __init__(self, name, chunk_list):
        self.name, self.chunks = name, chunk_list
        self.texts = [chunk_text_for_index(c) for c in chunk_list]
        sync()
        started = time.perf_counter()
        self.vectors = np.asarray(embedder.encode(self.texts, normalize_embeddings=True, batch_size=16, show_progress_bar=False), dtype=np.float32)
        sync()
        self.embed_seconds = time.perf_counter() - started
        self.faiss = faiss.IndexFlatIP(self.vectors.shape[1])
        self.faiss.add(self.vectors)
        self.bm25 = BM25Okapi([tokenize(t) for t in self.texts])


indexes = {name: Index(name, chunk_list) for name, chunk_list in chunk_sets.items()}
display(pd.DataFrame([{"індекс": name, "фрагментів": len(ix.chunks), "ембедингів": ix.faiss.ntotal, "розмірність": ix.vectors.shape[1],
                       "розмір матриці, МБ": ix.vectors.nbytes / 2 ** 20, "час ембедингів, с": ix.embed_seconds} for name, ix in indexes.items()]).round(1))

,індекс,фрагментів,ембедингів,розмірність,"розмір матриці, МБ","час ембедингів, с"
0,fixed-1000,935,935,1024,3.7,30.9
1,struct-1000,994,994,1024,3.9,30.6
2,struct-500,1936,1936,1024,7.6,31.0


## 1.4. Ретрив: hit@k і MRR для кількох конфігурацій

**Режими пошуку** (кожен повертає 10 найкращих фрагментів):

| режим | як працює |
|---|---|
| `dense` | запит кодується моделлю ембедингів, у FAISS беруться найближчі за скалярним добутком фрагменти |
| `bm25` | лексичний пошук за збігом слів (без лематизації); українські слова не збігаються з англійським текстом, тож працюють лише ідентифікатори й числа в питанні |
| `hybrid` | по 20 кандидатів від `dense` і `bm25` зливаються за **RRF** (Reciprocal Rank Fusion, сума 1/(60 + позиція)) |
| `hybrid+rerank` | 20 кандидатів гібридного пошуку переранжовує cross-encoder bge-reranker-v2-m3 |
| `hybrid+translate+rerank` | перед пошуком модель-генератор **перекладає питання англійською**, пошук виконується і за оригіналом, і за перекладом (dense за обома, BM25 за словами обох), переранжування за англійським запитом |

Останній режим націлений на головну складність цього корпусу: питання українською, а документація англійською, до того ж ключові терміни (`processor_kwargs`, `encode_query`) у питанні залишаються латиницею, а решта слів ні.

**Метрики** (лише для 25 питань із відповіддю; питання без відповіді тут не оцінюються, бо еталонного фрагмента не мають):

- **hit@k** — частка питань, для яких серед перших k знайдено **еталонний фрагмент** (фрагмент із еталонного файлу, що містить вираз-доказ). Це строга метрика на рівні фрагмента; для порівняння окремо виводиться hit@5 на рівні документа (є будь-який фрагмент еталонного файлу).
- **MRR@10** — середнє 1/позиція першого еталонного фрагмента (0, якщо його немає в перших 10).
- довірчий інтервал **Вілсона** 95% для hit@5: на 25 питаннях різниця менш ніж у кілька питань не є статистично значущою.

In [11]:
RRF_K = 60
CANDIDATES = 20
TOP_K = 10

TRANSLATE_SYSTEM = ("Ти перекладаєш питання про бібліотеку Sentence Transformers з української англійською. "
                    "Назви параметрів, методів, класів, моделей і числа залиш без змін. Виведи лише переклад, одним реченням.")
TRANSLATIONS = {}


def translate_query(question):
    if question not in TRANSLATIONS:
        TRANSLATIONS[question] = generate([{"role": "system", "content": TRANSLATE_SYSTEM}, {"role": "user", "content": question}], max_new_tokens=80)
    return TRANSLATIONS[question]


def embed_query(query):
    return np.asarray(embedder.encode([query], prompt_name="query", normalize_embeddings=True), dtype=np.float32)


def dense_ranking(ix, query, n=CANDIDATES):
    scores, ids = ix.faiss.search(embed_query(query), n)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]


def bm25_ranking(ix, query, n=CANDIDATES):
    scores = ix.bm25.get_scores(tokenize(query))
    order = [int(i) for i in np.argsort(-scores)[:n] if scores[i] > 0]  # фрагменти без жодного збігу слів у рейтинг не потрапляють
    return [(i, float(scores[i])) for i in order]


def rrf(rankings, k=RRF_K):
    fused = {}
    for ranking in rankings:
        for rank, (i, _) in enumerate(ranking, start=1):
            fused[i] = fused.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda pair: -pair[1])


def rerank(ix, query, candidates, n=TOP_K):
    scores = reranker.predict([(query, ix.texts[i]) for i, _ in candidates], batch_size=8, show_progress_bar=False)
    ranked = sorted(zip((i for i, _ in candidates), scores), key=lambda pair: -pair[1])
    return [(int(i), float(s)) for i, s in ranked[:n]]


RETRIEVAL_CACHE = {}


def retrieve(question, index_name, mode, k=TOP_K):
    # Повертає k найкращих фрагментів: [{"idx", "score", "chunk"}]; результат кешується
    key = (question, index_name, mode)
    if key not in RETRIEVAL_CACHE:
        ix = indexes[index_name]
        if mode == "dense":
            ranking = dense_ranking(ix, question, TOP_K)
        elif mode == "bm25":
            ranking = bm25_ranking(ix, question, TOP_K)
        elif mode == "hybrid":
            ranking = rrf([dense_ranking(ix, question), bm25_ranking(ix, question)])[:TOP_K]
        elif mode == "hybrid+rerank":
            candidates = rrf([dense_ranking(ix, question), bm25_ranking(ix, question)])[:CANDIDATES]
            ranking = rerank(ix, question, candidates)
        elif mode == "hybrid+translate+rerank":
            english = translate_query(question)
            lexical_query = " ".join(dict.fromkeys(tokenize(question) + tokenize(english)))
            candidates = rrf([dense_ranking(ix, question), dense_ranking(ix, english), bm25_ranking(ix, lexical_query)])[:CANDIDATES]
            ranking = rerank(ix, english, candidates)
        else:
            raise ValueError(mode)
        RETRIEVAL_CACHE[key] = [{"idx": i, "score": s, "chunk": ix.chunks[i]} for i, s in ranking]
    return RETRIEVAL_CACHE[key][:k]


def reference_rank(results, item, level="chunk"):
    # Позиція першого еталонного результату (з 1) або None; level="chunk" — фрагмент із доказом, "doc" — будь-який фрагмент еталонного файлу
    for rank, result in enumerate(results, start=1):
        hit = matches_evidence(result["chunk"], item) if level == "chunk" else result["chunk"]["doc_id"] in item["gold"]
        if hit:
            return rank
    return None


def wilson(successes, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = successes / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return centre - half, centre + half

In [12]:
CONFIGS = {
    "A": ("fixed-1000", "dense"),
    "B": ("struct-1000", "dense"),
    "C": ("struct-500", "dense"),
    "D": ("struct-1000", "bm25"),
    "E": ("struct-1000", "hybrid"),
    "F": ("struct-1000", "hybrid+rerank"),
    "G": ("struct-1000", "hybrid+translate+rerank"),
    "H": ("fixed-1000", "hybrid+rerank"),
    "I": ("struct-500", "hybrid+rerank"),
}
CONFIG_TITLES = {name: f"{name}: {index_name} · {mode}" for name, (index_name, mode) in CONFIGS.items()}

chunk_ranks, doc_ranks, retrieval_seconds, context_chars = {}, {}, {}, {}
for name, (index_name, mode) in CONFIGS.items():
    sync()
    started = time.perf_counter()
    results = {item["id"]: retrieve(item["question"], index_name, mode) for item in answerable_items}
    sync()
    retrieval_seconds[name] = (time.perf_counter() - started) / len(answerable_items)
    chunk_ranks[name] = {item["id"]: reference_rank(results[item["id"]], item) for item in answerable_items}
    doc_ranks[name] = {item["id"]: reference_rank(results[item["id"]], item, "doc") for item in answerable_items}
    context_chars[name] = np.mean([len(r["chunk"]["text"]) for res in results.values() for r in res[:5]])
    print(f"{CONFIG_TITLES[name]}: готово, {retrieval_seconds[name] * 1000:.0f} мс на питання")

print("\nПереклад питань англійською (приклади):")
for item in answerable_items[:5]:
    print(f"  {item['question']}\n    → {TRANSLATIONS[item['question']]}")

A: fixed-1000 · dense: готово, 48 мс на питання
B: struct-1000 · dense: готово, 48 мс на питання
C: struct-500 · dense: готово, 56 мс на питання
D: struct-1000 · bm25: готово, 4 мс на питання
E: struct-1000 · hybrid: готово, 67 мс на питання
F: struct-1000 · hybrid+rerank: готово, 350 мс на питання
G: struct-1000 · hybrid+translate+rerank: готово, 2819 мс на питання
H: fixed-1000 · hybrid+rerank: готово, 412 мс на питання
I: struct-500 · hybrid+rerank: готово, 224 мс на питання

Переклад питань англійською (приклади):
  Яку мінімальну версію Python рекомендують для Sentence Transformers?
    → Яку мінімальну версію Python рекомендують для Sentence Transformers?
  Яку версію бібліотеки transformers вимагає Sentence Transformers v6.0?
    → Sentence Transformers v6.0 вимагає версію бібліотеки transformers 4.33.0.
  Як після оновлення до v5.4+ називається метод get_sentence_embedding_dimension() у SentenceTransformer?
    → Після оновлення до v5.4+ метод get_sentence_embedding_dimension()

In [13]:
N = len(answerable_items)
summary_rows = []
for name in CONFIGS:
    ranks = list(chunk_ranks[name].values())
    row = {"конфігурація": CONFIG_TITLES[name]}
    for k in (1, 3, 5, 10):
        row[f"hit@{k}"] = sum(r is not None and r <= k for r in ranks) / N
    row["MRR@10"] = sum(1 / r for r in ranks if r is not None) / N
    low, high = wilson(sum(r is not None and r <= 5 for r in ranks), N)
    row["hit@5, 95% CI"] = f"{low:.2f}–{high:.2f}"
    row["hit@5 (документ)"] = sum(r is not None and r <= 5 for r in doc_ranks[name].values()) / N
    row["мс/питання"] = retrieval_seconds[name] * 1000
    row["сер. довжина топ-5, симв."] = context_chars[name]
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).set_index("конфігурація")
display(summary.round(2))

BEST = max(CONFIGS, key=lambda name: (summary.loc[CONFIG_TITLES[name], "MRR@10"], summary.loc[CONFIG_TITLES[name], "hit@5"]))
BEST_INDEX, BEST_MODE = CONFIGS[BEST]
print(f"\nНайкраща за MRR@10 конфігурація: {CONFIG_TITLES[BEST]} (MRR {summary.loc[CONFIG_TITLES[BEST], 'MRR@10']:.2f})")

,hit@1,hit@3,hit@5,hit@10,MRR@10,"hit@5, 95% CI",hit@5 (документ),мс/питання,"сер. довжина топ-5, симв."
конфігурація,,,,,,,,,
A: fixed-1000 · dense,0.56,0.72,0.76,0.88,0.66,0.57–0.89,0.92,47.57,973.75
B: struct-1000 · dense,0.56,0.84,0.92,0.96,0.70,0.75–0.98,1.00,48.04,725.18
C: struct-500 · dense,0.52,0.76,0.84,0.92,0.65,0.65–0.94,0.96,56.20,382.62
D: struct-1000 · bm25,0.32,0.52,0.56,0.76,0.44,0.37–0.73,0.80,4.24,745.88
E: struct-1000 · hybrid,0.56,0.68,0.80,0.92,0.66,0.61–0.91,0.92,66.79,718.81
F: struct-1000 · hybrid+rerank,0.68,0.92,0.96,0.96,0.80,0.80–0.99,1.00,349.79,757.13
G: struct-1000 · hybrid+translate+rerank,0.68,0.92,0.96,0.96,0.80,0.80–0.99,1.00,2818.90,756.47
H: fixed-1000 · hybrid+rerank,0.76,0.88,0.92,0.92,0.82,0.75–0.98,1.00,412.34,996.15
I: struct-500 · hybrid+rerank,0.68,0.88,0.96,1.00,0.80,0.80–0.99,1.00,224.37,401.90



Найкраща за MRR@10 конфігурація: H: fixed-1000 · hybrid+rerank (MRR 0.82)


In [14]:
# Позиція еталонного фрагмента в видачі для кожного питання: «—» означає, що його немає в топ-10
rank_table = pd.DataFrame({name: {item["id"]: chunk_ranks[name][item["id"]] or "—" for item in answerable_items} for name in CONFIGS})
rank_table.insert(0, "тип", [item["kind"] for item in answerable_items])
display(rank_table)

by_kind = []
for name in ("A", BEST):
    for kind in sorted({item["kind"] for item in answerable_items}):
        ids = [item["id"] for item in answerable_items if item["kind"] == kind]
        ranks = [chunk_ranks[name][i] for i in ids]
        by_kind.append({"конфігурація": CONFIG_TITLES[name], "тип питання": kind, "питань": len(ids),
                        "hit@5": sum(r is not None and r <= 5 for r in ranks) / len(ids), "MRR@10": sum(1 / r for r in ranks if r is not None) / len(ids)})
display(pd.DataFrame(by_kind).round(2))

,тип,A,B,C,D,E,F,G,H,I
q01,число,1,1,1,2,1,1,1,1,1
q02,число,1,5,—,1,1,1,1,1,3
q03,ідентифікатор,1,2,4,1,1,2,1,1,2
q04,ідентифікатор,2,1,2,1,1,1,1,1,1
q05,ідентифікатор,1,1,1,1,1,1,1,2,1
q06,концепція,1,1,1,1,1,1,1,1,1
q07,число,3,3,5,3,4,2,2,3,7
q08,число,—,9,10,6,6,2,1,1,2
q09,концепція,—,3,3,—,5,1,1,—,1
q10,концепція,1,2,3,9,2,1,1,1,1


,конфігурація,тип питання,питань,hit@5,MRR@10
0,A: fixed-1000 · dense,концепція,9,0.78,0.79
1,A: fixed-1000 · dense,число,6,0.83,0.59
2,A: fixed-1000 · dense,ідентифікатор,10,0.70,0.58
3,H: fixed-1000 · hybrid+rerank,концепція,9,0.89,0.89
4,H: fixed-1000 · hybrid+rerank,число,6,1.00,0.89
5,H: fixed-1000 · hybrid+rerank,ідентифікатор,10,0.90,0.72


**Висновки за ретривом.** Усе нижче — на 25 питаннях із відповіддю; hit@k і MRR@10 — за *еталонним фрагментом*. При n = 25 довірчі інтервали Вілсона для hit@5 мають півширину приблизно 0,1–0,2, тож різниця в одне-два питання (0,04–0,08) — це шум.

1. **Розбиття без переранжування.** Для dense-пошуку `struct-1000` (B) краще за вікна фіксованої довжини (A): hit@5 0,92 проти 0,76, MRR 0,70 проти 0,66; дрібніші `struct-500` (C) гірші за B (0,84 і 0,65). На рівні *файлу* B завжди має потрібний документ у топ-5 (1,00), A — у 0,92: у чотирьох питань A знаходить правильний файл, але не потрібний фрагмент. Різниця B–A — чотири питання, інтервали Вілсона ([0,75; 0,98] і [0,57; 0,89]) перекриваються: це тенденція, а не доведений факт.
2. **BM25 окремо слабкий** (D: hit@5 0,56, MRR 0,44). Питання українською, документація англійською, тож збігаються лише слова, що в питанні лишилися латиницею. На 1-ше місце BM25 ставить правильний фрагмент у 8 питаннях із 25 (hit@1 = 0,32): це питання з назвами параметрів і класів (q03–q05) або з англійськими термінами (q06, q12, q23, q25: CausalLM, cross-encoder, all-MiniLM-L6-v2, DDP).
3. **Гібрид (RRF) без переранжування dense не покращив:** E проти B — hit@5 0,80 проти 0,92, MRR 0,66 проти 0,70; краще у 3 питаннях (q02, q03, q08), гірше у 5 (q07, q09, q13, q17, q19). Імовірне пояснення: слабкий BM25-список із п. 2 вносить у злиття сторонні фрагменти нарівні з dense, а RRF не знає, що цьому списку треба довіряти менше (окремо я цього не перевіряв). Порівняно з самим BM25 гібрид, навпаки, значно кращий (E краще за D у 12 питаннях, гірше в 1).
4. **Головний виграш дає переранжування.** F проти E: hit@1 0,68 проти 0,56, hit@5 0,96 проти 0,80, MRR 0,80 проти 0,66 (краще у 9 питаннях, гірше у 2). Cross-encoder багатомовний, тож добре працює зі зв'язкою «питання українською — фрагмент англійською». Ціна: 350 мс на питання проти 67 мс у гібриду (у 5 разів).
5. **З переранжуванням розбиття майже не розрізняються.** H (`fixed-1000`): MRR 0,82, hit@1 0,76; F (`struct-1000`) і I (`struct-500`): по 0,80 і 0,68; H проти F — по 4 питання в кожен бік. H стала `BEST` лише за формальним критерієм «найбільший MRR@10», а перевага 0,02 — це шум. Різниця в іншому: I — єдина з усіх, що знаходить усі 25 питань у топ-10 (hit@10 = 1,00), і найшвидша з переранжованих (224 мс, бо фрагменти коротші), а H не має q09 у топ-10 взагалі, хоча F, G і I ставлять його на 1-ше місце.
6. **Конфігурація G (переклад питання) не перевірила переклад.** Метрики G збігаються з F (MRR 0,80), але G у 8 разів повільніша (2819 мс проти 350 мс — через виклик генератора). Надруковані вище приклади показують причину: у 5 показаних випадках **жодного англійського речення**. У трьох модель не переклала питання, а відповіла на нього, до того ж хибно («transformers 4.33.0» замість v5.x; «tokenizer_model» замість `processing_class`), а ці вигадані «відповіді» потрапили в пошуковий запит; у двох повторила питання українською майже дослівно. Ймовірна причина — моя помилка в системному промпті перекладу (там написано «з української англійською» замість «з української на англійську»), підсилена слабкою інструкційною дисципліною 4-мільярдної моделі; перевірити це можна лише повторним запуском. Тому висновок «переклад не допомагає» робити **не можна**: G фактично не перевіряла переклад.
7. **За типами питань** (A → H): числа — hit@5 0,83 → 1,00, MRR 0,59 → 0,89; назви параметрів і методів — 0,70 → 0,90 і 0,58 → 0,72; концепції — 0,78 → 0,89 і 0,79 → 0,89. Питання про ідентифікатори найважчі навіть для найкращої конфігурації.
8. **Питання q18** (який batch sampler радять для in-batch negatives) не потрапляє в топ-10 у 8 із 9 конфігурацій; лише I знаходить його (4-те місце). Той самий абзац дублюється в `training_overview.md` усіх чотирьох типів моделей (тому в еталоні чотири файли), а поруч є схожі за темою фрагменти `loss_overview.md`.

## 1.5. Генерація з цитуванням і відмовою

**Модель** — Qwen3-4B-Instruct-2507, жадібне декодування, ліміт 350 нових токенів. У промпт потрапляють перші **k = 5** фрагментів видачі, кожен з номером `[n]` і шляхом «файл > розділ». Системний промпт вимагає: відповідати лише за фрагментами, **ставити номер фрагмента після кожного твердження**, а коли відповіді у фрагментах немає, **відмовлятися** фіксованою фразою.

**Три прогони на всіх 30 питаннях:**

1. **без RAG** — ті самі питання, без жодних фрагментів (чи знає модель документацію версії 6.3.0 сама);
2. **RAG базовий** — конфігурація A (фрагменти фіксованої довжини, лише dense-пошук): те, що дає демо без доробок;
3. **RAG остаточний** — найкраща за MRR конфігурація з розділу 1.4.

**Як оцінюється правильність відповіді.**

- **За правилами (основна метрика).** Для питання з відповіддю відповідь правильна, якщо вона не є відмовою і містить усі ключові факти питання (`facts`: назви, числа). Для питання без відповіді правильна лише відмова. Метрика прозора й відтворювана, але груба: вона не помітить правильного факту з вигаданим контекстом і може пропустити помилку, якщо ключове слово вжито в іншому сенсі.
- **Суддя-LLM (перевірка).** Та сама модель порівнює відповідь з еталонною. Це **слабке місце**: модель-суддя і модель-генератор збігаються, а така пара схильна завищувати оцінку власним відповідям (self-preference). Тому суддя не підміняє правила, а лише перехресно перевіряє їх: рахуємо узгодженість, перевіряємо суддю на завідомо правильних і завідомо хибних відповідях, а розбіжності розбираємо вручну.

**Як оцінюється цитування.** Номери `[n]` у відповіді зіставляються з фрагментами промпту:

- *цитує еталонний файл* — принаймні один процитований фрагмент походить з еталонного документа;
- *цитує еталонний фрагмент* — принаймні один процитований фрагмент містить вираз-доказ (строга перевірка: то справді те місце документації, де є відповідь);
- *вигадані номери* — посилання на номер, якого немає в промпті.

Цитування оцінюється лише для питань з відповіддю, на які система не відмовилась. Для питань без відповіді правильна поведінка — відмова без жодних посилань.

In [15]:
REFUSAL = "У наданих документах немає відповіді."

SYSTEM_PROMPT_V1 = (
    "Ти асистент, що відповідає на питання про бібліотеку Sentence Transformers за її документацією.\n"
    "Правила:\n"
    "1. Відповідай українською, стисло: 1–3 речення. Назви параметрів, методів, класів, моделей і числа переписуй точно, як у фрагментах.\n"
    "2. Використовуй лише надані фрагменти. Власних знань про бібліотеку не використовуй.\n"
    "3. Після кожного твердження став номер фрагмента, з якого воно взяте, у квадратних дужках: [1] або [2].\n"
    f"4. Якщо у фрагментах немає відповіді на питання, відповідай лише так: «{REFUSAL}»"
)
NO_RAG_SYSTEM = ("Ти асистент з бібліотеки Sentence Transformers. Відповідай українською стисло, 1–3 речення. "
                 "Якщо не знаєш відповіді, прямо скажи, що не знаєш.")


def fragment_header(chunk):
    return chunk["doc_id"] + (" > " + chunk["section"] if chunk["section"] else "")


def build_messages(question, results, system_prompt):
    context = "\n\n".join(f"[{n}] {fragment_header(r['chunk'])}\n{r['chunk']['text']}" for n, r in enumerate(results, start=1))
    return [{"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Фрагменти документації:\n\n{context}\n\nПитання: {question}"}]


REFUSAL_PATTERN = re.compile(r"немає відповіді|не знаю|не маю (?:достатньо )?(?:інформації|даних)|немає (?:достатньо )?(?:інформації|даних)"
                             r"|не можу (?:відповісти|знайти)|не місти\w+", re.I)
CITATION = re.compile(r"\[(\d+(?:\s*,\s*\d+)*)\]")


def is_refusal(answer):
    return bool(REFUSAL_PATTERN.search(answer[:150]))


def cited_numbers(answer):
    return sorted({int(n) for group in CITATION.findall(answer) for n in re.findall(r"\d+", group)})


def prompt_tokens(messages):
    return len(tokenizer(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))["input_ids"])


example = ITEM_BY_ID["q03"]
example_messages = build_messages(example["question"], retrieve(example["question"], "struct-1000", "dense")[:2], SYSTEM_PROMPT_V1)
print("Приклад промпту (2 фрагменти замість 5, щоб не займати місця):\n")
print(tokenizer.apply_chat_template(example_messages, add_generation_prompt=True, tokenize=False)[:2600], "…")

Приклад промпту (2 фрагменти замість 5, щоб не займати місця):

<|im_start|>system
Ти асистент, що відповідає на питання про бібліотеку Sentence Transformers за її документацією.
Правила:
1. Відповідай українською, стисло: 1–3 речення. Назви параметрів, методів, класів, моделей і числа переписуй точно, як у фрагментах.
2. Використовуй лише надані фрагменти. Власних знань про бібліотеку не використовуй.
3. Після кожного твердження став номер фрагмента, з якого воно взяте, у квадратних дужках: [1] або [2].
4. Якщо у фрагментах немає відповіді на питання, відповідай лише так: «У наданих документах немає відповіді.»<|im_end|>
<|im_start|>user
Фрагменти документації:

[1] migration_guide.md > Migration Guide > Migrating from v5.x to v5.4+ > Renamed methods and parameters
       | Also :meth:`~sentence_transformers.sentence_transformer.model.SentenceTransformer.encode_query`, :meth:`~sentence_transformers.sentence_transformer.model.SentenceTransformer.encode_document`, the corresponding :cla

In [16]:
def judge(item, answer):
    reference = item["reference"] or "Відповіді в документації немає. Правильна поведінка: відмовитися відповідати (написати, що відповіді немає чи що не знаєш)."
    prompt = ("Ти перевіряєш відповіді на питання про бібліотеку Sentence Transformers.\n"
              f"Питання: {item['question']}\nЕталонна відповідь: {reference}\nВідповідь для перевірки: {answer}\n\n"
              "Чи збігається відповідь для перевірки з еталонною за змістом (ті самі ключові назви, числа й факти, без суперечностей)? "
              "Відповідай одним словом: ТАК або НІ.")
    return generate([{"role": "user", "content": prompt}], max_new_tokens=5).strip().upper().startswith("ТАК")


def run_pipeline(label, items, index_name=None, mode=None, k=5, system_prompt=SYSTEM_PROMPT_V1):
    # index_name=None — прогін без RAG
    rows = []
    for item in items:
        results = retrieve(item["question"], index_name, mode)[:k] if index_name else []
        messages = build_messages(item["question"], results, system_prompt) if index_name else \
            [{"role": "system", "content": NO_RAG_SYSTEM}, {"role": "user", "content": item["question"]}]
        sync()
        started = time.perf_counter()
        answer = generate(messages)
        sync()
        rows.append({"id": item["id"], "answer": answer, "gen_seconds": time.perf_counter() - started,
                     "prompt_tokens": prompt_tokens(messages), "results": results})
    return {"label": label, "k": k, "rag": bool(index_name), "rows": rows}


def evaluate_run(run):
    for row in run["rows"]:
        item, answer, results = ITEM_BY_ID[row["id"]], row["answer"], row["results"]
        row["refused"] = is_refusal(answer)
        if item["gold"]:
            row["correct"] = (not row["refused"]) and all(re.search(f, answer, re.S) for f in item["facts"])
        else:
            row["correct"] = row["refused"]
        row["judge"] = judge(item, answer)
        numbers = cited_numbers(answer)
        valid = [n for n in numbers if 1 <= n <= len(results)]
        row["cited"] = numbers
        row["invalid_numbers"] = [n for n in numbers if n not in valid]
        row["cites_gold_doc"] = any(results[n - 1]["chunk"]["doc_id"] in item["gold"] for n in valid)
        row["cites_evidence"] = any(matches_evidence(results[n - 1]["chunk"], item) for n in valid)
        row["context_rank"] = reference_rank(results, item) if item["gold"] else None
    return run


def summarize_run(run):
    rows = run["rows"]
    with_answer = [r for r in rows if ITEM_BY_ID[r["id"]]["gold"]]
    without_answer = [r for r in rows if not ITEM_BY_ID[r["id"]]["gold"]]
    answered = [r for r in with_answer if not r["refused"]]
    correct_total = sum(r["correct"] for r in rows)
    low, high = wilson(correct_total, len(rows))

    def share(count):
        return count / len(answered) if answered and run["rag"] else float("nan")

    return {
        "варіант": run["label"],
        "правильно, усього (з 30)": correct_total,
        "95% CI": f"{low:.2f}–{high:.2f}",
        "з відповіддю (з 25)": sum(r["correct"] for r in with_answer),
        "правильні відмови (з 5)": sum(r["correct"] for r in without_answer),
        "хибні відмови (з 25)": sum(r["refused"] for r in with_answer),
        "суддя: правильно (з 30)": sum(r["judge"] for r in rows),
        "є цитування": share(sum(bool(r["cited"]) for r in answered)),
        "цитує еталонний файл": share(sum(r["cites_gold_doc"] for r in answered)),
        "цитує еталонний фрагмент": share(sum(r["cites_evidence"] for r in answered)),
        "вигадані номери (відповідей)": sum(bool(r["invalid_numbers"]) for r in rows),
        "правильно і з еталонним фрагментом (з 25)": sum(r["correct"] and r["cites_evidence"] for r in with_answer),
        "сер. генерація, с": np.mean([r["gen_seconds"] for r in rows]),
        "сер. токенів у промпті": np.mean([r["prompt_tokens"] for r in rows]),
    }


def cohen_kappa(a, b):
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    expected = pa * pb + (1 - pa) * (1 - pb)
    return (observed - expected) / (1 - expected) if expected < 1 else float("nan")

In [17]:
runs = {}
runs["без RAG"] = evaluate_run(run_pipeline("без RAG", BENCHMARK))
print("без RAG: готово")
runs["RAG базовий"] = evaluate_run(run_pipeline(f"RAG базовий ({CONFIG_TITLES['A']}, k=5)", BENCHMARK, *CONFIGS["A"]))
print("RAG базовий: готово")
runs["RAG остаточний"] = evaluate_run(run_pipeline(f"RAG остаточний ({CONFIG_TITLES[BEST]}, k=5)", BENCHMARK, BEST_INDEX, BEST_MODE))
print("RAG остаточний: готово")

generation_summary = pd.DataFrame([summarize_run(run) for run in runs.values()]).set_index("варіант")
display(generation_summary.T.round(2))

без RAG: готово
RAG базовий: готово
RAG остаточний: готово


варіант,без RAG,"RAG базовий (A: fixed-1000 · dense, k=5)","RAG остаточний (H: fixed-1000 · hybrid+rerank, k=5)"
"правильно, усього (з 30)",2,25,26
95% CI,0.02–0.21,0.66–0.93,0.70–0.95
з відповіддю (з 25),2,20,22
правильні відмови (з 5),0,5,4
хибні відмови (з 25),3,2,0
суддя: правильно (з 30),5,22,25
є цитування,NaN,1.0,1.0
цитує еталонний файл,NaN,0.913043,1.0
цитує еталонний фрагмент,NaN,0.826087,0.84
вигадані номери (відповідей),1,0,0


In [18]:
# Результат для кожного питання в остаточному прогоні
rows_by_run = {label: {row["id"]: row for row in run["rows"]} for label, run in runs.items()}
table = []
for item in BENCHMARK:
    row = rows_by_run["RAG остаточний"][item["id"]]
    table.append({
        "id": item["id"], "тип": item["kind"],
        "без RAG": "✓" if rows_by_run["без RAG"][item["id"]]["correct"] else "✗",
        "базовий": "✓" if rows_by_run["RAG базовий"][item["id"]]["correct"] else "✗",
        "остаточний": "✓" if row["correct"] else "✗",
        "відмова": "так" if row["refused"] else "",
        "еталонний фрагмент у промпті, позиція": row["context_rank"] or "—",
        "цитує": ", ".join(f"[{n}]" for n in row["cited"]) or "—",
        "цитує еталонний фрагмент": "так" if row["cites_evidence"] else ("" if row["refused"] else "ні"),
    })
display(pd.DataFrame(table).set_index("id"))

kind_rows = []
for kind in ("число", "ідентифікатор", "концепція", "без відповіді"):
    ids = [item["id"] for item in BENCHMARK if item["kind"] == kind]
    kind_rows.append({"тип питання": kind, "питань": len(ids),
                      **{label: f"{sum(rows_by_run[label][i]['correct'] for i in ids)}/{len(ids)}" for label in runs}})
display(pd.DataFrame(kind_rows).set_index("тип питання"))

,тип,без RAG,базовий,остаточний,відмова,"еталонний фрагмент у промпті, позиція",цитує,цитує еталонний фрагмент
id,,,,,,,,
q01,число,✗,✓,✓,,1,[1],так
q02,число,✗,✓,✓,,1,[1],так
q03,ідентифікатор,✗,✓,✓,,1,[1],так
q04,ідентифікатор,✗,✓,✓,,1,[1],так
q05,ідентифікатор,✗,✓,✓,,2,[2],так
q06,концепція,✗,✓,✓,,1,[1],так
q07,число,✗,✓,✓,,3,[3],так
q08,число,✗,✗,✓,,1,[2],ні
q09,концепція,✗,✗,✗,,—,[1],ні


,питань,без RAG,RAG базовий,RAG остаточний
тип питання,,,,
число,6,0/6,5/6,6/6
ідентифікатор,10,1/10,9/10,9/10
концепція,9,1/9,6/9,7/9
без відповіді,5,0/5,5/5,4/5


In [19]:
# 1) Узгодженість «правила ↔ суддя» на всіх 90 відповідях трьох прогонів
rule_labels = [row["correct"] for run in runs.values() for row in run["rows"]]
judge_labels = [row["judge"] for run in runs.values() for row in run["rows"]]
print(pd.crosstab(pd.Series(rule_labels, name="правила"), pd.Series(judge_labels, name="суддя")))
agreement = sum(a == b for a, b in zip(rule_labels, judge_labels)) / len(rule_labels)
print(f"Узгодженість: {agreement:.2f}, Cohen's kappa: {cohen_kappa(rule_labels, judge_labels):.2f}")

# 2) Контролі судді: еталонні відповіді мають отримати «ТАК», відповіді на чужі питання — «НІ»
positive = [judge(item, item["reference"] or REFUSAL) for item in BENCHMARK]
negative = []
for j, item in enumerate(BENCHMARK):
    wrong = next(other for other in (answerable_items[(j + shift) % len(answerable_items)] for shift in range(7, 40)) if other["id"] != item["id"])
    negative.append(judge(item, wrong["reference"]))
print(f"\nКонтроль «еталон → ТАК»: {sum(positive)}/{len(positive)}")
print(f"Контроль «відповідь на чуже питання → НІ»: {len(negative) - sum(negative)}/{len(negative)}")

# 3) Розбіжності для ручного розбору
print("\nРозбіжності правил і судді:")
for label, run in runs.items():
    for row in run["rows"]:
        if row["correct"] != row["judge"]:
            item = ITEM_BY_ID[row["id"]]
            print(f"[{label}] {item['id']}: правила={'✓' if row['correct'] else '✗'}, суддя={'✓' if row['judge'] else '✗'}")
            print(f"   еталон: {item['reference'] or '(відмова)'}")
            print(f"   відповідь: {row['answer'][:300]}")

суддя    False  True 
правила              
False       32      5
True         6     47
Узгодженість: 0.88, Cohen's kappa: 0.75

Контроль «еталон → ТАК»: 30/30
Контроль «відповідь на чуже питання → НІ»: 30/30

Розбіжності правил і судді:
[без RAG] q12: правила=✗, суддя=✓
   еталон: Він рахується для кожної пари запит–текст, тому повільніший; його використовують, щоб переранжувати top-k результатів bi-encoder.
   відповідь: Cross-encoder повільніший, бо для кожного підходу обчислює повну відповідність між двома текстами, а не просто використовує вектори. Зазвичай використовується для завдань, де потрібна висока точність, наприклад, у пошуку відповідей або класифікації пар текстів.
[без RAG] q13: правила=✗, суддя=✓
   еталон: Завантажити CrossEncoder з activation_fn=torch.nn.Sigmoid(); на ранжування це не впливає.
   відповідь: MS MARCO cross-encoder не має вбудованої можливості повертати оцінки від 0 до 1 — він використовується для визначення схожості пари (запит-відповідь) і повертає о

**Висновки за генерацією.** 30 питань: 25 із відповіддю, 5 без. Основна метрика — правила за ключовими фактами, суддя-LLM — перехресна перевірка.

**1. RAG потрібен.** Без документів правильних відповідей 2 із 25 (числа — 0 із 6, назви — 1 із 10, концепції — 1 із 9), а на 5 питаннях без відповіді модель **жодного разу не відмовилась** (0 із 5: вигадала відповідь на кожне). Три хибні відмови «не знаю» були на питаннях, відповідь на які в корпусі є. Одна з двох «правильних» відповідей без RAG насправді хибна: на питання про залежність для `use_kernels=True` модель назвала `torch`, еталон — `sentence-transformers[kernels]`; правило схвалило відповідь, бо слово `kernels` є в самому питанні, а суддя відхилив. З RAG: 25 із 30 (базовий, A) і 26 із 30 (остаточний, H); різниця в одне питання, інтервали Вілсона [0,66; 0,93] і [0,70; 0,95] перекриваються. Вирішальний виграш дає саме RAG (2 → 25–26), а не вибір конфігурації.

**2. Базовий проти остаточного.** Остаточний прибрав хибні відмови (2 → 0) і цитує еталонний файл у 100 % відповідей проти 91 %; цитування еталонного *фрагмента* майже не змінилось (84 % проти 83 %). Ціна: на питаннях без відповіді правильних відмов 4 із 5 проти 5 із 5 (q27, див. 1.6).

**3. Ретрив — вузьке місце.** В остаточному прогоні еталонний фрагмент потрапив у промпт у 23 із 25 питань (не потрапив у q09 і q18). Коли він у промпті, правильно відповіла модель на 22 із 23 питань за правилами (виняток q12); коли ні — на 0 із 2. Одну з цих 22 відповідей (q23) за моїм розбором слід вважати хибною, див. п. 6.

**4. Цитування.** Усі відповіді мають посилання, вигаданих номерів немає (без RAG така відповідь одна). У 100 % відповідей процитовано фрагмент із правильного файлу, але лише у 84 % (21 із 25) — той фрагмент, де є вираз-доказ. Чотири «промахи» — q08 і q20 (відповідь правильна, але процитовано інший фрагмент того самого файлу) і q09 та q18 (еталонного фрагмента в промпті немає). Метрика сувора: див. 1.6.

**5. За типами питань** (без RAG → базовий → остаточний): числа 0/6 → 5/6 → 6/6; назви 1/10 → 9/10 → 9/10; концепції 1/9 → 6/9 → 7/9; без відповіді 0/5 → 5/5 → 4/5. Промпт зростає з 115 токенів (без RAG) до ≈ 1500 (з п'ятьма фрагментами); генерація — 4,1–4,6 с на відповідь проти 5,6 с без RAG.

**6. Суддя й правила.** На 90 відповідях (3 прогони × 30) правила й суддя збіглися в 79 випадках (узгодженість 0,88, κ Коена 0,75); розбіжностей 11 (5 разів правила ✗ / суддя ✓, 6 разів навпаки). Контролі судді пройдено: еталонна відповідь → «ТАК» 30 із 30, відповідь на чуже питання → «НІ» 30 із 30 (але це лише грубі випадки). Усі 11 розбіжностей я розібрав вручну:

- **правила помилилися (4):** q16 без RAG (див. п. 1); q18 базовий (у тексті є `NO_DUPLICATES`, але головна рекомендація — інший sampler, суддя правий); q23 остаточний (відповідь суперечлива: каже, що швидша «в 5 разів», і закінчує «У наданих документах немає відповіді», правило не розпізнало відмову, бо шукає її лише на початку); q27 остаточний (по суті відмова, див. 1.6);
- **суддя помилився (6):** q13 у трьох прогонах (відповідь `activation_fn=torch.nn.Sigmoid()` правильна, і в базовому та остаточному прогонах суддя її відхилив, ймовірно через відсутність «на ранжування це не впливає»; а в прогоні без RAG схвалив відповідь, що суперечить еталону); q24 базовий (названо evaluator і FULL_STATE_DICT, суддя ✗); q28 без RAG (суддя схвалив відповідь на питання без відповіді замість відмови); q12 без RAG (пояснює повільність, але не згадує переранжування, суддя поблажливий);
- **спірний (1):** q25 без RAG (загальна відповідь «ефективніший», еталон вимагає конкретних причин).

Отже, у розбіжностях правила помилялися рідше за суддю (4 проти 6), але жодна метрика не бездоганна. Напрям помилок судді цікавий: у прогоні без RAG він м'якший за правила (5 «правильних» проти 2), у прогонах із RAG суворіший (22 проти 25 і 25 проти 26). Це узгоджується з побоюванням щодо упередженості до власних відповідей (self-preference), але на 4–5 випадках це лише натяк, а не доказ; з іншим суддею я не порівнював. Підсумок за ручною корекцією двох помилок метрики в остаточному прогоні (q23 −1, q27 +1): 26 із 30, як і за правилами.

## 1.6. Аналіз відмов

Розбираємо кожну відповідь остаточного прогону, яка не пройшла перевірку, і відносимо її до одного з **п'яти типів**:

| тип | ознака |
|---|---|
| **Промах ретриву** | еталонного фрагмента немає серед фрагментів у промпті (позиція «—»); навіть правильна відповідь у такому разі не підтверджена джерелом |
| **Хибна відмова** | еталонний фрагмент у промпті, але модель відповіла «немає відповіді» |
| **Помилка генерації** | еталонний фрагмент у промпті, але відповідь неправильна чи неповна |
| **Галюцинація** | питання без відповіді, але модель відповіла замість відмови |
| **Помилка цитування** | відповідь правильна, але жоден процитований фрагмент не є еталонним |

Тип присвоюється кодом за позицією еталонного фрагмента в промпті, відмовою й правильністю відповіді. Це **попередня** класифікація: причину кожного випадку (чому саме так сталося) визначаємо вручну за розбором нижче, а також перевіряємо, чи не помилилася сама метрика (правила проти судді).

In [20]:
FAILURE_TYPES = {
    "retrieval": "промах ретриву",
    "false_refusal": "хибна відмова",
    "generation": "помилка генерації",
    "hallucination": "галюцинація",
    "citation": "помилка цитування",
}


def failure_type(item, row):
    # None — відповідь пройшла всі перевірки
    if not item["gold"]:
        return None if row["refused"] else "hallucination"
    if row["context_rank"] is None:
        return "retrieval"
    if row["refused"]:
        return "false_refusal"
    if not row["correct"]:
        return "generation"
    if not row["cites_evidence"]:
        return "citation"
    return None


type_counts = pd.DataFrame({label: Counter(failure_type(ITEM_BY_ID[r["id"]], r) or "ok" for r in run["rows"]) for label, run in runs.items()})
type_counts = type_counts.reindex(list(FAILURE_TYPES) + ["ok"]).fillna(0).astype(int)
type_counts = type_counts.rename(index={**FAILURE_TYPES, "ok": "без відмов"})
display(type_counts)

final_rows = {row["id"]: row for row in runs["RAG остаточний"]["rows"]}
failing_ids = [item["id"] for item in BENCHMARK if failure_type(item, final_rows[item["id"]])]
print(f"Відмов в остаточному прогоні: {len(failing_ids)} із {len(BENCHMARK)}\n")
for item_id in failing_ids:
    item, row = ITEM_BY_ID[item_id], final_rows[item_id]
    kind = failure_type(item, row)
    top = [fragment_header(r["chunk"]) for r in row["results"][:3]]
    print(f"{item_id} [{item['kind']}] {item['question']}")
    print(f"  еталон: {item['reference'] or '(відмова: відповіді в корпусі немає)'}")
    print(f"  відповідь: {row['answer'][:420]}")
    print(f"  еталонний файл: {', '.join(item['gold']) or '—'}; фрагмент у промпті на позиції: {row['context_rank'] or '—'}; цитує: {row['cited'] or '—'}")
    print(f"  топ-3 у промпті: {top}")
    print(f"  правила: {'✓' if row['correct'] else '✗'}, суддя: {'✓' if row['judge'] else '✗'}")
    print(f"  → попередній тип: {FAILURE_TYPES[kind]}\n")

,без RAG,RAG базовий,RAG остаточний
промах ретриву,25,6,2
хибна відмова,0,0,0
помилка генерації,0,1,1
галюцинація,5,0,1
помилка цитування,0,0,2
без відмов,0,23,24


Відмов в остаточному прогоні: 6 із 30

q08 [число] Яке медіанне прискорення порівняно з FP32 дає FP16 разом із Flash Attention та unpadding?
  еталон: Приблизно у 3,87 раза (медіана за протестованими моделями).
  відповідь: Медіанне прискорення порівняно з FP32 при FP16 разом із Flash Attention та unpadding — 3.87x. [2]
  еталонний файл: sentence_transformer/usage/efficiency.rst; фрагмент у промпті на позиції: 1; цитує: [2]
  топ-3 у промпті: ['sentence_transformer/usage/efficiency.rst', 'sentence_transformer/usage/efficiency.rst', 'sparse_encoder/usage/efficiency.rst']
  правила: ✓, суддя: ✓
  → попередній тип: помилка цитування

q09 [концепція] Які бекенди для обчислення ембедингів підтримує Sentence Transformers?
  еталон: Три бекенди: PyTorch (типовий), ONNX і OpenVINO.
  відповідь: Sentence Transformers підтримує бекенди, такі як llama.cpp та вбудовані реалізації (наприклад, FP32, BF16), але конкретні варіанти, такі як FA2 або llama.cpp, можуть бути обмежені для певних моделей. На

In [21]:
# Чи можна відсікати питання без відповіді за порогом релевантності? Дивимося на оцінку найкращого фрагмента
top1 = {item["id"]: (retrieve(item["question"], BEST_INDEX, BEST_MODE) or [{"score": 0.0}])[0]["score"] for item in BENCHMARK}
pos = [top1[item["id"]] for item in answerable_items]
neg = [top1[item["id"]] for item in unanswerable_items]
auc = np.mean([[p > n for n in neg] for p in pos])
print(f"Оцінка найкращого фрагмента ({CONFIG_TITLES[BEST]}):")
print(f"  питання з відповіддю: мін. {min(pos):.2f}, медіана {np.median(pos):.2f}, макс. {max(pos):.2f}")
neg_text = ", ".join(f"{item['id']}={top1[item['id']]:.2f}" for item in unanswerable_items)
print(f"  питання без відповіді: {neg_text}")
print(f"  ймовірність, що випадкове питання з відповіддю має вищу оцінку, ніж випадкове без відповіді (AUC): {auc:.2f}")
best_threshold, best_accuracy = None, -1
for threshold in sorted(set(pos + neg)):
    accuracy = (sum(p >= threshold for p in pos) + sum(n < threshold for n in neg)) / len(BENCHMARK)
    if accuracy > best_accuracy:
        best_threshold, best_accuracy = threshold, accuracy
wrongly_cut = sum(p < best_threshold for p in pos)
print(f"  найкращий поріг {best_threshold:.2f}: правильно розділяє {best_accuracy:.0%} питань, відсікає {wrongly_cut} із {len(pos)} питань з відповіддю")

Оцінка найкращого фрагмента (H: fixed-1000 · hybrid+rerank):
  питання з відповіддю: мін. 0.71, медіана 0.97, макс. 1.00
  питання без відповіді: q26=0.82, q27=0.97, q28=0.17, q29=0.25, q30=0.57
  ймовірність, що випадкове питання з відповіддю має вищу оцінку, ніж випадкове без відповіді (AUC): 0.87
  найкращий поріг 0.71: правильно розділяє 93% питань, відсікає 0 із 25 питань з відповіддю


### Виправлення

Три експерименти на остаточній конфігурації пошуку, кожен повторює повний прогін усіх 30 питань і перевіряється тими самими метриками:

1. **Суворіший промпт (v2)**: окремий крок «знайди місце, що прямо відповідає», пряма вказівка, що схожа тема, інша модель чи інше число не є відповіддю, вимога цитувати саме той фрагмент, де написано твердження. Мета: прибрати галюцинації на «близьких» питаннях і неточне цитування.
2. **Більше фрагментів (k = 8)**: мета: повернути у промпт еталонний фрагмент, який ретрив поставив нижче п'ятої позиції. Ціна: довший промпт і більше відволікаючих фрагментів.
3. **Обидва разом**.

In [22]:
SYSTEM_PROMPT_V2 = (
    "Ти асистент, що відповідає на питання про бібліотеку Sentence Transformers лише за наданими фрагментами документації.\n"
    "Порядок роботи:\n"
    "1. Знайди у фрагментах місце, яке прямо відповідає на питання. Схожа тема, інша модель, інше число GPU чи інша версія — це не відповідь.\n"
    f"2. Якщо такого місця немає, відповідай лише так: «{REFUSAL}» Без пояснень і без посилань на фрагменти.\n"
    "3. Інакше відповідай українською, стисло: 1–3 речення. Назви параметрів, методів, класів, моделей і числа переписуй точно, як у фрагменті.\n"
    "4. Після кожного твердження став номер фрагмента у квадратних дужках: [1]. Цитуй той фрагмент, у якому справді написано саме це твердження.\n"
    "5. Не додавай нічого, чого немає у фрагментах."
)
PROMPTS = {"v1": SYSTEM_PROMPT_V1, "v2": SYSTEM_PROMPT_V2}
VARIANTS = {"v2, k=5": ("v2", 5), "v1, k=8": ("v1", 8), "v2, k=8": ("v2", 8)}

experiments = {"v1, k=5 (остаточний)": runs["RAG остаточний"]}
for label, (prompt_name, k) in VARIANTS.items():
    experiments[label] = evaluate_run(run_pipeline(label, BENCHMARK, BEST_INDEX, BEST_MODE, k=k, system_prompt=PROMPTS[prompt_name]))
    print(f"{label}: готово")

experiment_summary = pd.DataFrame([{**summarize_run(run), "варіант": label} for label, run in experiments.items()]).set_index("варіант")
display(experiment_summary[["правильно, усього (з 30)", "95% CI", "з відповіддю (з 25)", "правильні відмови (з 5)", "хибні відмови (з 25)",
                            "суддя: правильно (з 30)", "цитує еталонний фрагмент", "правильно і з еталонним фрагментом (з 25)",
                            "сер. генерація, с", "сер. токенів у промпті"]].round(2))

base_failing = set(failing_ids)
effect_rows = []
for item_id in failing_ids:
    item = ITEM_BY_ID[item_id]
    row = {"id": item_id, "тип": FAILURE_TYPES[failure_type(item, final_rows[item_id])]}
    for label, run in list(experiments.items())[1:]:
        new = next(r for r in run["rows"] if r["id"] == item_id)
        kind = failure_type(item, new)
        row[label] = "✓ виправлено" if kind is None else f"✗ ({FAILURE_TYPES[kind]})"
    effect_rows.append(row)
display(pd.DataFrame(effect_rows).set_index("id"))

failures_total, regressions = {}, {}
for label, run in experiments.items():
    failed = {r["id"] for r in run["rows"] if failure_type(ITEM_BY_ID[r["id"]], r)}
    failures_total[label], regressions[label] = len(failed), sorted(failed - base_failing)
display(pd.DataFrame({"відмов усього": failures_total, "нові відмови (регресії)": {k: ", ".join(v) or "—" for k, v in regressions.items()}}))

IMPROVED = min(experiments, key=lambda label: (failures_total[label], len(regressions[label])))
IMPROVED_PROMPT_NAME, IMPROVED_K = ("v1", 5) if IMPROVED.startswith("v1, k=5") else VARIANTS[IMPROVED]
IMPROVED_PROMPT = PROMPTS[IMPROVED_PROMPT_NAME]
print(f"\nНайменше відмов: «{IMPROVED}» ({failures_total[IMPROVED]}); цей варіант використовується в інтерфейсі.")

v2, k=5: готово
v1, k=8: готово
v2, k=8: готово


,"правильно, усього (з 30)",95% CI,з відповіддю (з 25),правильні відмови (з 5),хибні відмови (з 25),суддя: правильно (з 30),цитує еталонний фрагмент,правильно і з еталонним фрагментом (з 25),"сер. генерація, с",сер. токенів у промпті
варіант,,,,,,,,,,
"v1, k=5 (остаточний)",26,0.70–0.95,22,4,0,25,0.84,20,4.60,1532.60
"v2, k=5",28,0.79–0.98,23,5,1,24,0.83,20,4.73,1619.60
"v1, k=8",26,0.70–0.95,21,5,1,24,0.83,19,4.41,2255.83
"v2, k=8",26,0.70–0.95,21,5,1,25,0.79,18,5.13,2342.83


,тип,"v2, k=5","v1, k=8","v2, k=8"
id,,,,
q08,помилка цитування,✗ (помилка цитування),✗ (помилка цитування),✗ (помилка цитування)
q09,промах ретриву,✗ (промах ретриву),✗ (промах ретриву),✗ (промах ретриву)
q12,помилка генерації,✓ виправлено,✗ (помилка генерації),✗ (помилка генерації)
q18,промах ретриву,✗ (промах ретриву),✗ (промах ретриву),✗ (промах ретриву)
q20,помилка цитування,✗ (помилка цитування),✗ (помилка цитування),✗ (помилка цитування)
q27,галюцинація,✓ виправлено,✓ виправлено,✓ виправлено


,відмов усього,нові відмови (регресії)
"v1, k=5 (остаточний)",6,—
"v2, k=5",5,q13
"v1, k=8",6,q23
"v2, k=8",7,"q13, q23"



Найменше відмов: «v2, k=5» (5); цей варіант використовується в інтерфейсі.


### Підсумкова таблиця відмов

Остаточний прогін (H · hybrid+rerank, промпт v1, k = 5): **6 відмов із 30** за кодом. Після ручного розбору п'ять із них — справжні помилки системи, а одна (q27) — помилка метрики. Експерименти: **v2** — суворіший промпт, **k=8** — вісім фрагментів у промпті.

| № | Питання | Тип за кодом | Що відбувалося насправді (ручний розбір) | Виправлення | Результат |
|---|---|---|---|---|---|
| 1 | q08 [число] медіанне прискорення FP16 + Flash Attention | помилка цитування | Відповідь правильна (3.87x; правила й суддя ✓). Процитовано [2], а фрагмент із виразом-доказом стояв у промпті 1-м; [2] — інший фрагмент того самого `efficiency.rst`. Чи є число й у [2], з виводу не видно: це або надто сувора метрика, або справді неточне цитування | v2: «цитуй той фрагмент, де справді написано твердження» | не виправлено в жодному з трьох експериментів |
| 2 | q09 [концепція] які бекенди підтримує бібліотека | промах ретриву | Фрагмент із «supports 3 backends» не потрапив у топ-10 конфігурації H (`fixed-1000`). Відповідь хибна: перелічує llama.cpp, FP32, BF16 замість PyTorch, ONNX, OpenVINO, і спирається на інший фрагмент `efficiency.rst`. Відмови не було, хоча еталонного фрагмента в промпті не було | k=8 (і v2) | не виправлено: еталонний фрагмент лежить поза топ-10, тож більше фрагментів у промпті не допоможе. Не перевірено наскрізно: конфігурації F, G, I ставлять цей фрагмент 1-м |
| 3 | q12 [концепція] чому cross-encoder повільніший | помилка генерації | Фрагмент 1-й у промпті, відповідь за змістом майже правильна (пари запит–текст, переранжування top-k), але мовно зіпсована: «перерангування», «швидше вимагає обчислень». Правило шукає «переранж…» і через опечатку відхилило; суддя теж ✗ | v2 | виправлено у v2 k=5; **не** виправлено в v1 k=8 і v2 k=8. Результат нестабільний і залежить від промпту й k; один жадібний запуск не доводить, що помилку усунено |
| 4 | q18 [ідентифікатор] який batch sampler радять для in-batch negatives | промах ретриву | Фрагмент про `BatchSamplers.NO_DUPLICATES` поза топ-10 у 8 із 9 конфігурацій (лише I ставить його 4-м). У промпті фрагменти `loss_overview.md`; відповідь загальна («BatchSamplers») без `NO_DUPLICATES` | k=8, v2 | не виправлено. Пропозиція (не перевірена): дрібніші фрагменти (конфігурація I знаходить фрагмент) або переформулювання запиту |
| 5 | q20 [ідентифікатор] який клас замінює `pylate.models.ColBERT` | помилка цитування | Відповідь правильна (`MultiVectorEncoder`), процитовано [1] (теж `migration_guide.md`), а фрагмент із виразом-доказом стоїть у промпті лише 5-м (dense-пошук A ставив його 1-м, а в конфігурації H · hybrid+rerank він 5-й). Імовірно, сувора метрика, як у q08 | v2 | не виправлено |
| 6 | q27 [без відповіді] скільки параметрів у all-mpnet-base-v2 | галюцинація (за кодом) | По суті **коректна, але розлога відмова**: модель написала, що кількість параметрів цієї моделі в документах не наведена, і зауважила, що 133M належить іншій моделі (mpnet-base). Вигаданого числа немає. Метрика не впізнала відмову (вона шукає фразу на початку відповіді) | v2: «відповідай лише так: … без пояснень і без посилань» | виправлено (за метрикою) у всіх трьох експериментах |

**Експерименти.** v2 при k=5: відмов 6 → 5 (виправлено q12 і q27, нова відмова q13), правильних 26 → 28 із 30, інтервал Вілсона [0,79; 0,98] проти [0,70; 0,95] перекривається. Збільшення k до 8 не допомагає: промпт довшає на 47–53 % (≈ 2250–2340 токенів проти 1530), відмов 6 і 7 (нові відмови q23 і q13), еталонний фрагмент цитується рідше (0,83 і 0,79 проти 0,84). Обрано «v2, k=5» (5 відмов); його використовує інтерфейс у 1.7. Варіант обрано за тими самими 30 питаннями, на яких він оцінюється, тож 28 із 30 — оптимістична оцінка.

**Чи допоможе поріг релевантності.** Оцінка найкращого фрагмента (cross-encoder) для питань із відповіддю — від 0,71 до 1,00 (медіана 0,97), для питань без відповіді: q26 = 0,82, q27 = 0,97, q28 = 0,17, q29 = 0,25, q30 = 0,57; AUC 0,87. Найкращий поріг 0,71 відсікає q28–q30 і не відсікає жодного питання з відповіддю, але пропускає q26 і q27 (оцінка q27 дорівнює медіані питань із відповіддю). Поріг підібрано на тих самих питаннях, тож 93 % точності — оптимістична оцінка; для «близьких» питань на кшталт q27 його недостатньо, і відмову мусить ухвалювати генератор за змістом фрагментів.

## 1.7. Інтерфейс

Функція `ask(question)` приймає питання українською і друкує відповідь та **джерела**: номер фрагмента, файл і розділ документації, оцінку пошуку, посилання на файл у репозиторії та початок тексту. Фрагменти, на які посилається відповідь, позначені `✔`. Використовується найкращий пошук з розділу 1.4 і найкращий за результатами розділу 1.6 варіант промпту.

Нижче — той самий конвеєр у вигляді веб-інтерфейсу Gradio (у Colab запускається в клітинці; якщо Gradio недоступний, достатньо `ask()`).

In [23]:
GITHUB_BLOB = f"{CORPUS_REPO}/blob/{CORPUS_TAG}/docs/"


def answer_question(question, k=None):
    k = k or IMPROVED_K
    results = retrieve(question, BEST_INDEX, BEST_MODE)[:k]
    answer = generate(build_messages(question, results, IMPROVED_PROMPT))
    return answer, results, cited_numbers(answer)


def format_sources(results, cited):
    lines = []
    for n, r in enumerate(results, start=1):
        snippet = " ".join(r["chunk"]["text"].split())[:170]
        lines.append(f"{'✔' if n in cited else ' '} [{n}] {fragment_header(r['chunk'])}  (оцінка {r['score']:.2f})\n"
                     f"      {GITHUB_BLOB}{r['chunk']['doc_id']}\n      «{snippet}…»")
    return "\n".join(lines)


def ask(question, k=None):
    answer, results, cited = answer_question(question, k)
    print(f"Питання: {question}\n\nВідповідь: {answer}\n")
    if is_refusal(answer):
        print("(Система відмовилась відповідати: у корпусі немає підстав. Найближчі фрагменти, на які відповідь НЕ спирається:)")
    else:
        print("Джерела (✔ — процитовано у відповіді):")
    print(format_sources(results, cited))
    print()


ask("Чим метод encode_query відрізняється від encode_document?")
ask("Як запустити Sentence Transformers на Raspberry Pi?")

Питання: Чим метод encode_query відрізняється від encode_document?

Відповідь: Метод `encode_query` використовує "query" prompt (якщо доступний) і задає `task="query"`, а метод `encode_document` використовує "document" prompt (якщо доступний) і задає `task="document"`. Це дозволяє моделям коректно обробляти запити та документи з різними інструкціями. [4]

Джерела (✔ — процитовано у відповіді):
  [1] migration_guide.md  (оцінка 0.95)
      https://github.com/huggingface/sentence-transformers/blob/v6.3.0/docs/migration_guide.md
      «se methods are specialized version of :meth:`~sentence_transformers.sentence_transformer.model.SentenceTransformer.encode` that differs in exactly two ways: 1. If no ``pr…»
  [2] migration_guide.md  (оцінка 0.94)
      https://github.com/huggingface/sentence-transformers/blob/v6.3.0/docs/migration_guide.md
      «"Paris is the capital of France." # The new encode_query and encode_document methods call encode, # but with the prompt name set to "query" or "do

In [24]:
try:
    import gradio as gr

    def gradio_ask(question):
        answer, results, cited = answer_question(question)
        sources = "\n\n".join(
            f"{'✔ ' if n in cited else ''}**[{n}] {fragment_header(r['chunk'])}** (оцінка {r['score']:.2f}) — [{r['chunk']['doc_id']}]({GITHUB_BLOB}{r['chunk']['doc_id']})\n\n"
            f"> {' '.join(r['chunk']['text'].split())[:300]}…" for n, r in enumerate(results, start=1))
        return answer, sources

    demo = gr.Interface(
        fn=gradio_ask,
        inputs=gr.Textbox(label="Питання про Sentence Transformers (українською)", lines=2),
        outputs=[gr.Markdown(label="Відповідь"), gr.Markdown(label="Джерела")],
        title="RAG: документація Sentence Transformers 6.3.0",
        examples=[["Який параметр замінив tokenizer_kwargs?"], ["Скільки параметрів має модель all-mpnet-base-v2?"]],
        cache_examples=False,
    )
    demo.launch(prevent_thread_lock=True)  # не блокує ноутбук: решта клітинок виконується далі
except Exception as error:  # Gradio може бути не встановлено або не підтримувати цю версію
    print(f"Веб-інтерфейс не запущено ({type(error).__name__}: {error}); працює консольна функція ask() вище.")

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://308d164203d44e9285.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 1.8. Звіт

Нижче — звіт (числа взято з таблиць цього прогону); докладніші розбори — у розділах 1.4–1.6.

# Звіт: RAG-система для питань про документацію Sentence Transformers

## 1. Корпус і чому потрібен RAG

Корпус — документація бібліотеки Sentence Transformers (каталог `docs/` репозиторію https://github.com/huggingface/sentence-transformers, тег `v6.3.0`, коміт `cddcaee3405a5efde83c1d98f89c6ebd6733dab6`), ліцензія Apache-2.0 (перевірено кодом за файлом `LICENSE`). 44 документи (27 `.md` і 17 `.rst`), 842 КБ вихідного тексту, 169 234 токени генеративної моделі. **Корпус не додано до репозиторію**: ноутбук клонує його з GitHub за тегом і комітом; ваг моделей і векторного індексу в репозиторії теж немає. Питання українською, документація англійською, тож це ще й перевірка міжмовного пошуку.

RAG потрібен, бо: (1) корпус не вкладається в промпт: лише KV-кеш 169 тис. токенів оцінено у 23 ГБ при 15 ГБ відеопамʼяті T4; (2) документація змінюється між версіями (у v5.4 і v6.0 перейменовано методи й параметри), донавчена модель застаріє; (3) відповідь треба підтверджувати джерелом; (4) модель без документів не знає версію 6.3.0: у прогоні без RAG правильних відповідей 2 із 25, і жодного разу вона не відмовилась на питаннях без відповіді (0 із 5).

## 2. Бенчмарк

30 власних питань за текстом документації: 6 про числа й версії, 10 про назви параметрів і методів, 9 про концепції, 5 без відповіді в корпусі (з них 3 «близькі»: у документах є схожа, але інша інформація). Для кожного питання задано еталонну відповідь, еталонні файли, вирази-докази для еталонного *фрагмента* та ключові факти; для питань без відповіді задано проби відсутності. Код перевіряє, що еталонні файли збігаються з файлами, де знайдено доказ, а проби відсутності нічого не знаходять у корпусі (вивід: «Перевірка бенчмарку: помилок немає»).

## 3. Індексація

Модель ембедингів — Qwen3-Embedding-0.6B (1024-вимірні нормалізовані ембединги, запит з інструкцією `query`); сховище — FAISS `IndexFlatIP` (точний пошук) і BM25 (`rank_bm25`) над тими самими фрагментами; переранжування — cross-encoder bge-reranker-v2-m3 для 20 кандидатів гібридного пошуку. Три розбиття: `fixed-1000` (вікно 1000 символів, перекриття 200) та `struct-1000`, `struct-500` (за структурою документа: заголовки, абзаци, блоки коду; у кожного фрагмента шлях «файл > розділ»). Кодування одного набору фрагментів — близько 31 с.

| розбиття | фрагментів | середня довжина, симв. |
|---|---|---|
| fixed-1000 | 935 | 981.49 |
| struct-1000 | 994 | 732.72 |
| struct-500 | 1936 | 375.65 |

## 4. Результати ретриву

hit@k і MRR@10 за еталонним фрагментом, 25 питань із відповіддю (дев'ять конфігурацій):

| конфігурація | hit@1 | hit@3 | hit@5 | hit@10 | MRR@10 | hit@5, 95% CI | hit@5 (документ) | мс/питання |
|---|---|---|---|---|---|---|---|---|
| A: fixed-1000 · dense | 0.56 | 0.72 | 0.76 | 0.88 | 0.66 | 0.57–0.89 | 0.92 | 47.57 |
| B: struct-1000 · dense | 0.56 | 0.84 | 0.92 | 0.96 | 0.70 | 0.75–0.98 | 1.00 | 48.04 |
| C: struct-500 · dense | 0.52 | 0.76 | 0.84 | 0.92 | 0.65 | 0.65–0.94 | 0.96 | 56.20 |
| D: struct-1000 · bm25 | 0.32 | 0.52 | 0.56 | 0.76 | 0.44 | 0.37–0.73 | 0.80 | 4.24 |
| E: struct-1000 · hybrid | 0.56 | 0.68 | 0.80 | 0.92 | 0.66 | 0.61–0.91 | 0.92 | 66.79 |
| F: struct-1000 · hybrid+rerank | 0.68 | 0.92 | 0.96 | 0.96 | 0.80 | 0.80–0.99 | 1.00 | 349.79 |
| G: struct-1000 · hybrid+translate+rerank | 0.68 | 0.92 | 0.96 | 0.96 | 0.80 | 0.80–0.99 | 1.00 | 2818.90 |
| H: fixed-1000 · hybrid+rerank | 0.76 | 0.88 | 0.92 | 0.92 | 0.82 | 0.75–0.98 | 1.00 | 412.34 |
| I: struct-500 · hybrid+rerank | 0.68 | 0.88 | 0.96 | 1.00 | 0.80 | 0.80–0.99 | 1.00 | 224.37 |

- Для dense-пошуку структурне розбиття `struct-1000` краще за фіксовані вікна (hit@5 0,92 проти 0,76), але різниця — чотири питання, інтервали Вілсона перекриваються.
- BM25 окремо слабкий (MRR 0,44): питання українською, документація англійською. Гібрид (RRF) без переранжування не кращий за dense (E: MRR 0,66; B: 0,70).
- Найбільше дає переранжування cross-encoder'ом: MRR 0,66 → 0,80 (E → F) ціною 350 мс на питання проти 67 мс.
- Найкраща за MRR@10 — H (`fixed-1000` · hybrid+rerank, 0,82), але її перевага над F, G, I (0,80) — шум; I єдина знаходить усі 25 питань у топ-10.
- Конфігурація G (переклад питання) **не перевірила переклад**: у показаних прикладах модель не переклала питання, а відповіла на нього. Ймовірна причина — помилка в промпті перекладу («з української англійською»). Висновку про користь чи марність перекладу зробити не можна.

## 5. Генеративна модель і результати відповідей

**Генератор** — локальна Qwen3-4B-Instruct-2507 через `transformers`, жадібне декодування (4-мільярдна модель у половинній точності, ≈ 8 ГБ). **Лабораторну 4 (OpenAI-сумісний сервер) не виконано, тому окремого сервера немає: це заміна, яку треба враховувати.** Інтерфейс `generate(messages)` повторює формат запиту до чат-моделі, тож його можна замінити викликом OpenAI-сумісного endpoint без змін решти коду. Наслідки заміни: слабша за серверні моделі генерація (приклад — мовна помилка «перерангування» в q12), модель-генератор і модель-суддя та перекладач — одна й та сама модель.

У промпт потрапляє по 5 фрагментів із номерами; модель має цитувати номери після кожного твердження й відмовлятися фіксованою фразою, якщо відповіді немає.

| варіант | правильно, усього (з 30) | з відповіддю (з 25) | правильні відмови (з 5) | хибні відмови (з 25) | суддя: правильно (з 30) | цитує еталонний файл | цитує еталонний фрагмент | сер. генерація, с |
|---|---|---|---|---|---|---|---|---|
| без RAG | 2 | 2 | 0 | 3 | 5 | — | — | 5.57 |
| RAG базовий (A: fixed-1000 · dense, k=5) | 25 | 20 | 5 | 2 | 22 | 0.91 | 0.83 | 4.10 |
| RAG остаточний (H: fixed-1000 · hybrid+rerank, k=5) | 26 | 22 | 4 | 0 | 25 | 1.00 | 0.84 | 4.60 |

- Без RAG правильних відповідей 2 із 25 (одна з них хибна, схвалена правилом помилково), відмов на питання без відповіді 0 із 5. З RAG — 25 і 26 із 30; різниця між базовим і остаточним у одне питання, а різниця між «без RAG» і RAG величезна.
- Коли еталонний фрагмент потрапив у промпт (23 із 25 питань), модель відповіла правильно в 22 випадках за правилами: **вузьке місце — ретрив, а не генерація**.
- Цитування: посилання є в усіх відповідях, вигаданих номерів немає; правильний файл процитовано в 100 % відповідей, правильний фрагмент — у 84 % (21 із 25).
- Суддя й правила: узгодженість 0,88, κ Коена 0,75 на 90 відповідях; контролі судді 30/30 і 30/30. З 11 розбіжностей за моїм розбором правила помилилися в 4, суддя в 6, один випадок спірний.

## 6. Аналіз відмов

В остаточному прогоні 6 відмов із 30 (тип за кодом):

|  | кількість |
|---|---|
| промах ретриву | 2 |
| хибна відмова | 0 |
| помилка генерації | 1 |
| галюцинація | 1 |
| помилка цитування | 2 |
| без відмов | 24 |

Повна таблиця «випадок → тип → причина → виправлення → результат» — у розділі 1.6. Коротко: два промахи ретриву (q09, q18 — фрагмент поза топ-10), дві помилки цитування (q08, q20 — відповідь правильна, процитовано інший фрагмент того самого файлу, імовірно сувора метрика), одна помилка генерації (q12 — мовна), один випадок «галюцинації» (q27), який насправді є коректною, але розлогою відмовою, тобто помилкою метрики. Результат виправлень:

| варіант | правильно, усього (з 30) | з відповіддю (з 25) | правильні відмови (з 5) | цитує еталонний фрагмент |
|---|---|---|---|---|
| v1, k=5 (остаточний) | 26 | 22 | 4 | 0.84 |
| v2, k=5 | 28 | 23 | 5 | 0.83 |
| v1, k=8 | 26 | 21 | 5 | 0.83 |
| v2, k=8 | 26 | 21 | 5 | 0.79 |

Суворіший промпт v2 при k = 5 прибрав дві відмови (q12, q27) і додав одну (q13); збільшення k до 8 не допомагає (промпт довший на ≈ 50 %, відмов 6–7).

## 7. Інтерфейс

Функція `ask(питання)` друкує відповідь і джерела: номер фрагмента, файл і розділ, оцінку пошуку, посилання на файл у репозиторії на тезі `v6.3.0` та початок тексту; процитовані фрагменти позначено ✔. Така сама логіка працює у веб-інтерфейсі Gradio, який у Colab запускається окремою клітинкою. Приклад у ноутбуці: на питання про `encode_query` і `encode_document` система відповідає з посиланням, а на питання про Raspberry Pi відмовляється й показує найближчі фрагменти, на які відповідь не спирається.

## 8. Висновки й обмеження

**Остаточна система:** H (`fixed-1000` · hybrid+rerank), промпт v2, k = 5; правильних відповідей 28 із 30 (інтервал Вілсона [0,79; 0,98]); без RAG — 2, базовий RAG — 25.

**Обмеження, про які варто пам'ятати:**

- 30 питань — мала вибірка, інтервали широкі; різниці в одне-два питання — шум.
- Конфігурацію H і промпт v2 обрано за тими самими питаннями, на яких їх оцінено, окремого тестового набору немає, тож цифри оптимістичні.
- Метрики не бездоганні: за ручним розбором правила помилилися в 4 розбіжностях, суддя в 6; в остаточному прогоні q23 зараховано хибно, а q27 не зараховано, хоча по суті правильна (разом 26 із 30, як і за правилами).
- Суддя, генератор і перекладач — одна модель; упередженість до власних відповідей не виміряна, є лише натяк (у прогоні без RAG суддя м'якший за правила).
- Конфігурація G не перевіряла переклад (помилка в промпті перекладу).
- Лабораторну 4 не виконано: генерація локальна.
- Результати отримано одним жадібним запуском на GPU; повторні запуски можуть дещо відрізнятися.

**Що я зробив би далі:** виправити промпт перекладу й повторити G; перевіряти цитування за наявністю твердження у процитованому фрагменті, а не за виразом-доказом; порівняти суддю з сильнішою моделлю іншого сімейства; розширити бенчмарк до 100+ питань із відокремленим тестовим набором; для питань на кшталт q18 спробувати дрібніші фрагменти й переформулювання запиту.